# Chẩn đoán Raw riêng và Raw trong Dual EMA/Raw

**Kaggle: bật GPU T4 và Internet, rồi Run All.** Không cần upload ZIP cũ hoặc cache.
Notebook chứa mã model/trainer thực tế lấy từ hai ZIP seed 2026 đã đối chiếu. Không thay công thức train.

- Build **một cache chung**, chỉ cho 20 mẫu đầu tiên theo đúng permutation của seed 2026 trên tập 6.000 mẫu.
- Giữ batch/micro-batch 1, kiến trúc 384, LR schedule dựa trên 6.000 mẫu × 10 epoch; dừng sau 20 optimizer steps.
- Chạy Raw/Raw và Dual/Dual lặp lại, Raw/Dual, trong các process mới, ở chế độ thường và deterministic (8 runs).
- Ghi hash **nội dung tensor**, trạng thái RNG, logits, gradient và trọng số từng bước. Không dùng hash pickle để suy ra tensor khác nhau.
- Gói log, source, manifest, traceback và comparison thành `raw_dual_diagnostic.zip`. Gửi ZIP này để phân tích.

Giới hạn: phép thử chỉ xét đoạn đầu trước validation, không tái hiện các epoch sau. Nếu chưa thấy lệch, tăng `STEPS` (tối đa 6000). Deterministic báo lỗi vẫn được đóng gói, không bỏ qua âm thầm.


In [ ]:
%pip install -q "numpy==2.0.2" "transformers==5.23.0" "spacy==3.8.18" "scikit-learn==1.9.1" tqdm
import subprocess,sys
subprocess.run([sys.executable,'-m','spacy','download','en_core_web_sm'],check=True)


In [ ]:
from pathlib import Path
import subprocess,sys,json,datetime,hashlib
STEPS=20
DEVICE='cuda'
assert 1 <= STEPS <= 6000
WORK=Path('/kaggle/working')/('raw_dual_diagnostic_'+datetime.datetime.now().strftime('%Y%m%d_%H%M%S'))
WORK.mkdir(parents=True,exist_ok=False)
REPO=WORK/'PRDetect-Research'
COMMIT='6ada3d6cb183c92462c17eb46d187d3ddbadbe85'
subprocess.run(['git','clone','https://github.com/tphan1420/PRDetect-Research.git',str(REPO)],check=True)
subprocess.run(['git','checkout','--detach',COMMIT],cwd=REPO,check=True)
assert hashlib.sha256((REPO/'building_graph.py').read_bytes()).hexdigest() == '197536eaea5a1f7b7ccb043c265c93246461493e1d0b1e2d706ef7f88353b505'
print('Output folder:',WORK)


Mã nguồn gốc được nhúng bên dưới để không phụ thuộc các trainer mới thay đổi trên nhánh repo.


In [ ]:
SOURCES={'train_v3.py': '"""Train the Quang V3 Core from legacy-domain graphs.\n\nNo test files are read during training or checkpoint selection.\n"""\n\nimport argparse\nimport copy\nimport gc\nimport hashlib\nimport json\nimport math\nfrom pathlib import Path\nimport time\n\nfrom evaluate_v3_quang import (REPO, TRAINING_DOMAINS, environment, positive_int,\n                               sha256, write_json)\n\n\ndef build_parser():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--variants", nargs="+", choices=("baseline",), default=["baseline"],\n                        help="Retained for CLI compatibility; V3 Core has one baseline variant.")\n    parser.add_argument("--seeds", type=int, nargs="+", default=[2025, 2026])\n    parser.add_argument("--graph-dir", type=Path, default=REPO / "graph_data")\n    parser.add_argument("--output-dir", type=Path, default=REPO / "result/v3_baseline/model")\n    parser.add_argument("--train-file", default="hc3_train")\n    parser.add_argument("--val-file", default="hc3_val")\n    parser.add_argument("--struct-mode", choices=("low", "high", "directed"), default="low")\n    parser.add_argument("--num-rel", type=int, default=0, help="0: infer from training graph only")\n    parser.add_argument("--hidden", type=positive_int, default=384)\n    parser.add_argument("--num-bases", type=positive_int, default=12)\n    parser.add_argument("--layers", type=positive_int, default=2)\n    parser.add_argument("--dropout", type=float, default=0.3)\n    parser.add_argument("--in-dropout", type=float, default=0.1)\n    parser.add_argument("--edge-drop", type=float, default=0.1)\n    parser.add_argument("--struct", type=int, choices=(0, 1), default=1)\n    parser.add_argument("--gfeat", type=int, choices=(0, 1), default=0)\n    parser.add_argument("--readout", choices=("pool3",), default="pool3")\n    parser.add_argument("--aug", choices=("none", "swap", "mask", "both"), default="swap")\n    parser.add_argument("--aug-max", type=float, default=0.35)\n    parser.add_argument("--ema", type=int, choices=(0, 1), default=1)\n    parser.add_argument("--ema-decay", type=float, default=0.998)\n    parser.add_argument("--lr", type=float, default=5e-4)\n    parser.add_argument("--wd", type=float, default=0.01)\n    parser.add_argument("--epochs", type=positive_int, default=10)\n    parser.add_argument("--patience", type=positive_int, default=10)\n    parser.add_argument("--batch-size", type=positive_int, default=32,\n                        help="Effective graphs per optimizer/EMA/scheduler step")\n    parser.add_argument("--micro-batch-size", type=positive_int, default=8,\n                        help="Graph donor pool size; use the same value in both variants")\n    parser.add_argument("--val-batch-size", "--eval-batch-size", dest="val_batch_size",\n                        type=positive_int, default=64,\n                        help="Also controls the donor pool of the fixed validation swap proxy")\n    parser.add_argument("--select", choices=("lossaug", "loss", "acc", "auc"), default="lossaug")\n    parser.add_argument("--amp", type=int, choices=(0, 1), default=0)\n    parser.add_argument("--device", choices=("cpu", "cuda"), default="cuda")\n    parser.add_argument("--skip-completed", action="store_true")\n    parser.add_argument("--overwrite", action="store_true")\n    return parser\n\n\ndef validate_args(args):\n    pairs = {(item["train_file"], item["val_file"]) for item in TRAINING_DOMAINS.values()}\n    if (args.train_file, args.val_file) not in pairs:\n        raise ValueError("Use a supported train/validation pair from TRAINING_DOMAINS")\n    if len(set(args.seeds)) != len(args.seeds) or len(set(args.variants)) != len(args.variants):\n        raise ValueError("Seeds/variants must not contain duplicates")\n    for name in ("dropout", "in_dropout", "edge_drop", "aug_max"):\n        if not 0 <= getattr(args, name) <= 1:\n            raise ValueError(f"{name} must be in [0,1]")\n    for name in ("lr",):\n        if not math.isfinite(getattr(args, name)) or getattr(args, name) <= 0:\n            raise ValueError(f"{name} must be finite and positive")\n    if not math.isfinite(args.wd) or args.wd < 0 or not 0 <= args.ema_decay < 1 or args.num_rel < 0:\n        raise ValueError("Invalid weight decay, EMA decay or relation count")\n    if args.micro_batch_size > args.batch_size:\n        raise ValueError("micro-batch-size must not exceed batch-size")\n\n\ndef effective_readout(args, variant):\n    return args.readout\n\n\ndef build_model(args, variant, R):\n    from model.QuangV3 import Net\n\n    kwargs = dict(d=args.hidden, nb=args.num_bases, L=args.layers, p=args.dropout,\n                  p_in=args.in_dropout, edge_drop=args.edge_drop,\n                  use_struct=bool(args.struct), use_gfeat=bool(args.gfeat),\n                  readout=effective_readout(args, variant), aug=args.aug,\n                  aug_max=args.aug_max, mode=args.struct_mode)\n    return Net(R, **kwargs)\n\n\ndef load_graph(path, source, R, mode):\n    """Validate original cache schema and exact source label order, then keep on CPU."""\n    import numpy as np\n    import torch\n    from model import QuangV3 as v3\n\n    original_load = v3._load_pickle\n\n    def checked_load(filename):\n        raw = original_load(filename)\n        if not isinstance(raw, dict):\n            raise ValueError("Use dictionary caches from building_graph.py")\n        if any(key in raw for key in ("encoder_metadata", "schema_version", "all_cls_embeddings",\n                                      "all_quality", "quality_features")):\n            raise ValueError("Use legacy V1 graphs, not aligned/V2 caches")\n        for x, edge, rel in zip(raw["all_token_embeddings"], raw["all_edge_index"], raw["all_edge_type"]):\n            edge, rel = torch.as_tensor(edge), torch.as_tensor(rel)\n            if x.ndim != 2 or x.shape[0] == 0 or x.shape[1] != 768:\n                raise ValueError("Invalid legacy node embeddings; do not use aligned/V2 graphs")\n            if edge.shape != (2, rel.numel()) or rel.numel() == 0:\n                raise ValueError("Invalid relation/edge shapes")\n            if int(rel.min()) < 0 or int(edge.min()) < 0 or int(edge.max()) >= x.shape[0]:\n                raise ValueError("Invalid relation/edge indices")\n            if R and int(rel.max()) >= R:\n                raise ValueError("Validation relation IDs exceed training R; refusing silent clipping")\n        sizes = [len(raw[k]) for k in ("all_token_embeddings", "all_edge_index", "all_edge_type", "y")]\n        if len(set(sizes)) != 1:\n            raise ValueError("Cache arrays have different numbers of samples")\n        return raw\n\n    v3._load_pickle = checked_load\n    try:\n        ds = v3.GraphSet(str(path), R=R or None, mode=mode, store_dev="cpu")\n    finally:\n        v3._load_pickle = original_load\n    expected = []\n    with source.open(encoding="utf-8") as stream:\n        for line in stream:\n            if line.strip():\n                label = json.loads(line)["label"]\n                expected.append(int(label) if isinstance(label, (int, float)) else int("human" in str(label).lower()))\n    if not np.array_equal(ds.y.numpy(), expected) or set(expected) != {0, 1}:\n        raise ValueError("Cache/source labels or counts differ; rebuild with building_graph.py")\n    return ds\n\n\ndef validation_loss(probabilities, truth):\n    import numpy as np\n\n    p = np.clip(probabilities, 1e-6, 1 - 1e-6)\n    return float(-(truth * np.log(p) + (1 - truth) * np.log(1 - p)).mean())\n\n\ndef atomic_save(payload, path):\n    import torch\n\n    temporary = path.with_name(path.name + ".tmp")\n    torch.save(payload, temporary)\n    temporary.replace(path)\n\n\ndef signature(args, variant, seed, data_hashes):\n    config = vars(args).copy()\n    for key in ("variants", "seeds", "output_dir", "graph_dir", "skip_completed", "overwrite"):\n        config.pop(key)\n    config["effective_readout"] = effective_readout(args, variant)\n    code = {str(p.relative_to(REPO)): sha256(p) for p in (\n        Path(__file__), REPO / "model/QuangV3.py", REPO / "building_graph.py", REPO / "dep_vocab.py",\n        REPO / "dep_vocab.json",\n    )}\n    record = dict(config=config, variant=variant, seed=seed, data=data_hashes,\n                  code_sha256=code, environment=environment())\n    digest = hashlib.sha256(json.dumps(record, sort_keys=True).encode()).hexdigest()\n    return digest, record\n\n\ndef completed_run(path, record_path, run_signature):\n    if not path.is_file() or not record_path.is_file():\n        return False\n    record = json.loads(record_path.read_text(encoding="utf-8"))\n    return (record.get("complete") is True and record.get("run_signature") == run_signature\n            and record.get("checkpoint_sha256") == sha256(path))\n\n\ndef train_one(args, variant, seed, train, val, run_signature, provenance):\n    import numpy as np\n    import torch\n    import torch.nn.functional as F\n    from model.QuangV3 import predict, set_seed\n    from evaluate_v3_quang import score_probabilities\n\n    device = torch.device(args.device)\n    set_seed(seed)\n    model = build_model(args, variant, train.R).to(device)\n    teacher = copy.deepcopy(model).eval() if args.ema else None\n    optimizer = torch.optim.AdamW(model.parameters(), lr=args.lr, weight_decay=args.wd)\n    steps_per_epoch = math.ceil(train.G / args.batch_size)\n    total = steps_per_epoch * args.epochs\n    warm = max(1, int(total * 0.05))\n    schedule = torch.optim.lr_scheduler.LambdaLR(\n        optimizer, lambda s: (s + 1) / warm if s < warm else\n        0.5 * (1 + math.cos(math.pi * (s - warm) / max(1, total - warm))) * 0.95 + 0.05,\n    )\n    amp = bool(args.amp) and device.type == "cuda"\n    scaler = torch.amp.GradScaler("cuda", enabled=amp)\n    gm = train.GF.float().mean(0).cpu()\n    gs = train.GF.float().std(0).cpu().clamp_min(1e-6)\n    checkpoint_path = args.output_dir / f"v3_{variant}_seed{seed}.pt"\n    record_path = checkpoint_path.with_suffix(".train.json")\n    saved_args = {k: str(v) if isinstance(v, Path) else v for k, v in vars(args).items()}\n    saved_args.update(seeds=[seed], cons=0.0, readout=effective_readout(args, variant))\n    history, best_score, best_epoch, bad, step = [], float("inf"), 0, 0, 0\n    start = time.time()\n    sampler_generator = torch.Generator(device="cpu").manual_seed(seed)\n    print(f"{variant} seed={seed} | R={train.R} mode={args.struct_mode} | "\n          f"effective batch={args.batch_size}, micro={args.micro_batch_size}", flush=True)\n\n    for epoch in range(1, args.epochs + 1):\n        model.train()\n        permutation = torch.randperm(train.G, generator=sampler_generator)\n        epoch_loss = 0.0\n        for offset in range(0, train.G, args.batch_size):\n            indices = permutation[offset:offset + args.batch_size]\n            optimizer.zero_grad(set_to_none=True)\n            for micro in indices.split(args.micro_batch_size):\n                batch = train.batch(micro, device, gm, gs)\n                weight = batch["B"] / indices.numel()\n                branch_weight = 0.5 if args.aug != "none" else 1.0\n                # Release clean activations BEFORE the augmented forward. The\n                # effective step still averages all examples/branches exactly.\n                with torch.autocast(device.type, dtype=torch.float16, enabled=amp):\n                    clean = model(batch)\n                    clean_loss = F.binary_cross_entropy_with_logits(clean.float(), batch["y"])\n                scaler.scale(weight * branch_weight * clean_loss).backward()\n                epoch_loss += branch_weight * float(clean_loss.detach()) * batch["B"]\n                del clean, clean_loss\n                if args.aug != "none":\n                    with torch.autocast(device.type, dtype=torch.float16, enabled=amp):\n                        noisy = model(batch, aug_mode=args.aug)\n                        noisy_loss = F.binary_cross_entropy_with_logits(noisy.float(), batch["y"])\n                    scaler.scale(weight * 0.5 * noisy_loss).backward()\n                    epoch_loss += 0.5 * float(noisy_loss.detach()) * batch["B"]\n                    del noisy, noisy_loss\n                del batch\n            scaler.unscale_(optimizer)\n            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)\n            previous_scale = scaler.get_scale()\n            scaler.step(optimizer)\n            scaler.update()\n            if scaler.get_scale() >= previous_scale:\n                schedule.step()\n                step += 1\n                if teacher is not None:\n                    decay = min(args.ema_decay, (1 + step) / (10 + step))\n                    with torch.no_grad():\n                        for target, current in zip(teacher.parameters(), model.parameters()):\n                            target.mul_(decay).add_(current.detach(), alpha=1 - decay)\n        evaluation = teacher if teacher is not None else model\n        p = predict(evaluation, val, device, gm, gs, bs=args.val_batch_size, amp=amp)\n        p_swap = predict(evaluation, val, device, gm, gs, bs=args.val_batch_size, amp=amp, aug_mode="swap")\n        truth = val.y.numpy().astype(int)\n        metrics = score_probabilities(truth, p, 0.5)\n        clean_loss, swap_loss = validation_loss(p, truth), validation_loss(p_swap, truth)\n        score = {"lossaug": clean_loss + swap_loss, "loss": clean_loss,\n                 "acc": -metrics["accuracy"], "auc": -metrics["auc"]}[args.select]\n        if not math.isfinite(epoch_loss) or not math.isfinite(score):\n            raise FloatingPointError("Non-finite loss; no completed checkpoint will be marked")\n        history.append(dict(epoch=epoch, train_loss=epoch_loss / train.G, val_loss=clean_loss,\n                            val_swap_loss=swap_loss, selection_score=score, **metrics))\n        improved = score < best_score - 1e-6\n        if improved:\n            raw_state = {\n                k: v.detach().cpu().clone()\n                for k, v in model.state_dict().items()\n            }\n            ema_state = {\n                k: v.detach().cpu().clone()\n                for k, v in evaluation.state_dict().items()\n            }\n            best_score, best_epoch, bad = score, epoch, 0\n            thresholds = np.linspace(0.05, 0.95, 91)\n            accuracies = np.array([((p >= threshold) == truth).mean() for threshold in thresholds])\n            threshold = float(np.median(thresholds[accuracies >= accuracies.max() - 1e-9]))\n            state = {k: v.detach().cpu().clone() for k, v in evaluation.state_dict().items()}\n            atomic_save(dict(\n                state=ema_state,raw_state=raw_state, args=saved_args, R=train.R, gm=gm, gs=gs, thr=threshold,\n                method=f"v3_{variant}", struct_mode=args.struct_mode, best_epoch=best_epoch,\n                training_complete=False,\n                run_signature=run_signature, provenance=provenance,\n            ), checkpoint_path)\n            del state\n        else:\n            bad += 1\n        write_json(record_path, dict(complete=False, run_signature=run_signature,\n                                    provenance=provenance, best_epoch=best_epoch, history=history))\n        print(f"Epoch {epoch:02d} | loss {epoch_loss/train.G:.4f} | val {clean_loss:.4f} "\n              f"swap {swap_loss:.4f} | acc {metrics[\'accuracy\']:.4f} | "\n              f"({\'best\' if improved else \'keep\'})", flush=True)\n        if bad >= args.patience:\n            break\n    finished = torch.load(checkpoint_path, map_location="cpu", weights_only=True)\n    finished["training_complete"] = True\n    atomic_save(finished, checkpoint_path)\n    del finished\n    write_json(record_path, dict(complete=True, run_signature=run_signature,\n                                checkpoint_sha256=sha256(checkpoint_path), provenance=provenance,\n                                best_epoch=best_epoch, selection=args.select, history=history,\n                                train_seconds=time.time() - start))\n    print(f"Saved: {checkpoint_path}", flush=True)\n    del model, teacher, optimizer, scaler, schedule\n    gc.collect()\n    if device.type == "cuda":\n        torch.cuda.empty_cache()\n\n\ndef main(argv=None):\n    args = build_parser().parse_args(argv)\n    validate_args(args)\n    import torch\n\n    if args.device == "cuda" and not torch.cuda.is_available():\n        raise RuntimeError("Enable a GPU, or explicitly use --device cpu")\n    args.graph_dir, args.output_dir = args.graph_dir.resolve(), args.output_dir.resolve()\n    args.output_dir.mkdir(parents=True, exist_ok=True)\n    domain = next(item for item in TRAINING_DOMAINS.values()\n                  if (item["train_file"], item["val_file"]) == (args.train_file, args.val_file))\n    sources = [REPO / domain["train_source"], REPO / domain["val_source"]]\n    graphs = [args.graph_dir / f"{name}.pkl" for name in (args.train_file, args.val_file)]\n    data_hashes = {str(p): sha256(p) for p in sources + graphs}\n    pending = []\n    for variant in args.variants:\n        for seed in args.seeds:\n            path = args.output_dir / f"v3_{variant}_seed{seed}.pt"\n            record_path = path.with_suffix(".train.json")\n            sig, provenance = signature(args, variant, seed, data_hashes)\n            if args.skip_completed and completed_run(path, record_path, sig):\n                print(f"Skip completed: {path.name}", flush=True)\n            elif (path.exists() or record_path.exists()) and not args.overwrite:\n                raise FileExistsError(f"Incomplete/different run: {path}. Use a new output directory "\n                                      "or --overwrite; a saved best epoch is not proof training finished.")\n            else:\n                pending.append((variant, seed, sig, provenance))\n    if not pending:\n        return\n    train = load_graph(graphs[0], sources[0], args.num_rel, args.struct_mode)\n    val = load_graph(graphs[1], sources[1], train.R, args.struct_mode)\n    for variant, seed, sig, provenance in pending:\n        train_one(args, variant, seed, train, val, sig, provenance)\n\n\nif __name__ == "__main__":\n    main()\n', 'train_v3_dual_best.py': '"""Train the Quang V3 Core from legacy-domain graphs.\n\nNo test files are read during training or checkpoint selection.\n"""\n\nimport argparse\nimport copy\nimport gc\nimport hashlib\nimport json\nimport math\nfrom pathlib import Path\nimport time\n\nfrom evaluate_v3_quang import (REPO, TRAINING_DOMAINS, environment, positive_int,\n                               sha256, write_json)\n\n\ndef build_parser():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--variants", nargs="+", choices=("baseline",), default=["baseline"],\n                        help="Retained for CLI compatibility; V3 Core has one baseline variant.")\n    parser.add_argument("--seeds", type=int, nargs="+", default=[2025, 2026])\n    parser.add_argument("--graph-dir", type=Path, default=REPO / "graph_data")\n    parser.add_argument("--output-dir", type=Path, default=REPO / "result/v3_baseline/model")\n    parser.add_argument("--train-file", default="gpt3.5_mixed_train_split")\n    parser.add_argument("--val-file", default="gpt3.5_mixed_val_split")\n    parser.add_argument("--struct-mode", choices=("low", "high", "directed"), default="low")\n    parser.add_argument("--num-rel", type=int, default=0, help="0: infer from training graph only")\n    parser.add_argument("--hidden", type=positive_int, default=384)\n    parser.add_argument("--num-bases", type=positive_int, default=12)\n    parser.add_argument("--layers", type=positive_int, default=2)\n    parser.add_argument("--dropout", type=float, default=0.3)\n    parser.add_argument("--in-dropout", type=float, default=0.1)\n    parser.add_argument("--edge-drop", type=float, default=0.1)\n    parser.add_argument("--struct", type=int, choices=(0, 1), default=1)\n    parser.add_argument("--gfeat", type=int, choices=(0, 1), default=0)\n    parser.add_argument("--readout", choices=("pool3",), default="pool3")\n    parser.add_argument("--aug", choices=("none", "swap", "mask", "both"), default="swap")\n    parser.add_argument("--aug-max", type=float, default=0.35)\n    parser.add_argument("--ema", type=int, choices=(0, 1), default=1)\n    parser.add_argument("--ema-decay", type=float, default=0.998)\n    parser.add_argument("--lr", type=float, default=5e-4)\n    parser.add_argument("--wd", type=float, default=0.01)\n    parser.add_argument("--epochs", type=positive_int, default=10)\n    parser.add_argument("--patience", type=positive_int, default=10)\n    parser.add_argument("--batch-size", type=positive_int, default=32,\n                        help="Effective graphs per optimizer/EMA/scheduler step")\n    parser.add_argument("--micro-batch-size", type=positive_int, default=8,\n                        help="Graph donor pool size; use the same value in both variants")\n    parser.add_argument("--val-batch-size", "--eval-batch-size", dest="val_batch_size",\n                        type=positive_int, default=64,\n                        help="Also controls the donor pool of the fixed validation swap proxy")\n    parser.add_argument("--select", choices=("lossaug", "loss", "acc", "auc"), default="lossaug")\n    parser.add_argument("--amp", type=int, choices=(0, 1), default=0)\n    parser.add_argument("--device", choices=("cpu", "cuda"), default="cuda")\n    parser.add_argument("--skip-completed", action="store_true")\n    parser.add_argument("--overwrite", action="store_true")\n    return parser\n\n\ndef validate_args(args):\n    pairs = {(item["train_file"], item["val_file"]) for item in TRAINING_DOMAINS.values()}\n    if (args.train_file, args.val_file) not in pairs:\n        raise ValueError("Use a supported train/validation pair from TRAINING_DOMAINS")\n    if len(set(args.seeds)) != len(args.seeds) or len(set(args.variants)) != len(args.variants):\n        raise ValueError("Seeds/variants must not contain duplicates")\n    for name in ("dropout", "in_dropout", "edge_drop", "aug_max"):\n        if not 0 <= getattr(args, name) <= 1:\n            raise ValueError(f"{name} must be in [0,1]")\n    for name in ("lr",):\n        if not math.isfinite(getattr(args, name)) or getattr(args, name) <= 0:\n            raise ValueError(f"{name} must be finite and positive")\n    if not math.isfinite(args.wd) or args.wd < 0 or not 0 <= args.ema_decay < 1 or args.num_rel < 0:\n        raise ValueError("Invalid weight decay, EMA decay or relation count")\n    if args.micro_batch_size > args.batch_size:\n        raise ValueError("micro-batch-size must not exceed batch-size")\n\n\ndef effective_readout(args, variant):\n    return args.readout\n\n\ndef build_model(args, variant, R):\n    from model.QuangV3 import Net\n\n    kwargs = dict(d=args.hidden, nb=args.num_bases, L=args.layers, p=args.dropout,\n                  p_in=args.in_dropout, edge_drop=args.edge_drop,\n                  use_struct=bool(args.struct), use_gfeat=bool(args.gfeat),\n                  readout=effective_readout(args, variant), aug=args.aug,\n                  aug_max=args.aug_max, mode=args.struct_mode)\n    return Net(R, **kwargs)\n\n\ndef load_graph(path, source, R, mode):\n    """Validate original cache schema and exact source label order, then keep on CPU."""\n    import numpy as np\n    import torch\n    from model import QuangV3 as v3\n\n    original_load = v3._load_pickle\n\n    def checked_load(filename):\n        raw = original_load(filename)\n        if not isinstance(raw, dict):\n            raise ValueError("Use dictionary caches from building_graph.py")\n        if any(key in raw for key in ("encoder_metadata", "schema_version", "all_cls_embeddings",\n                                      "all_quality", "quality_features")):\n            raise ValueError("Use legacy V1 graphs, not aligned/V2 caches")\n        for x, edge, rel in zip(raw["all_token_embeddings"], raw["all_edge_index"], raw["all_edge_type"]):\n            edge, rel = torch.as_tensor(edge), torch.as_tensor(rel)\n            if x.ndim != 2 or x.shape[0] == 0 or x.shape[1] != 768:\n                raise ValueError("Invalid legacy node embeddings; do not use aligned/V2 graphs")\n            if edge.shape != (2, rel.numel()) or rel.numel() == 0:\n                raise ValueError("Invalid relation/edge shapes")\n            if int(rel.min()) < 0 or int(edge.min()) < 0 or int(edge.max()) >= x.shape[0]:\n                raise ValueError("Invalid relation/edge indices")\n            if R and int(rel.max()) >= R:\n                raise ValueError("Validation relation IDs exceed training R; refusing silent clipping")\n        sizes = [len(raw[k]) for k in ("all_token_embeddings", "all_edge_index", "all_edge_type", "y")]\n        if len(set(sizes)) != 1:\n            raise ValueError("Cache arrays have different numbers of samples")\n        return raw\n\n    v3._load_pickle = checked_load\n    try:\n        ds = v3.GraphSet(str(path), R=R or None, mode=mode, store_dev="cpu")\n    finally:\n        v3._load_pickle = original_load\n    expected = []\n    with source.open(encoding="utf-8") as stream:\n        for line in stream:\n            if line.strip():\n                label = json.loads(line)["label"]\n                expected.append(int(label) if isinstance(label, (int, float)) else int("human" in str(label).lower()))\n    if not np.array_equal(ds.y.numpy(), expected) or set(expected) != {0, 1}:\n        raise ValueError("Cache/source labels or counts differ; rebuild with building_graph.py")\n    return ds\n\n\ndef validation_loss(probabilities, truth):\n    import numpy as np\n\n    p = np.clip(probabilities, 1e-6, 1 - 1e-6)\n    return float(-(truth * np.log(p) + (1 - truth) * np.log(1 - p)).mean())\n\n\ndef atomic_save(payload, path):\n    import torch\n\n    temporary = path.with_name(path.name + ".tmp")\n    torch.save(payload, temporary)\n    temporary.replace(path)\n\n\ndef signature(args, variant, seed, data_hashes):\n    config = vars(args).copy()\n    for key in ("variants", "seeds", "output_dir", "graph_dir", "skip_completed", "overwrite"):\n        config.pop(key)\n    config["effective_readout"] = effective_readout(args, variant)\n    code = {str(p.relative_to(REPO)): sha256(p) for p in (\n        Path(__file__), REPO / "model/QuangV3.py", REPO / "building_graph.py", REPO / "dep_vocab.py",\n        REPO / "dep_vocab.json",\n    )}\n    record = dict(config=config, variant=variant, seed=seed, data=data_hashes,\n                  code_sha256=code, environment=environment())\n    digest = hashlib.sha256(json.dumps(record, sort_keys=True).encode()).hexdigest()\n    return digest, record\n\n\ndef completed_run(path, record_path, run_signature):\n    if not path.is_file() or not record_path.is_file():\n        return False\n    record = json.loads(record_path.read_text(encoding="utf-8"))\n    return (record.get("complete") is True and record.get("run_signature") == run_signature\n            and record.get("checkpoint_sha256") == sha256(path))\n\n\ndef train_one(args, variant, seed, train, val, run_signature, provenance):\n    import numpy as np\n    import torch\n    import torch.nn.functional as F\n    from model.QuangV3 import predict, set_seed\n    from evaluate_v3_quang import score_probabilities\n\n    device = torch.device(args.device)\n    set_seed(seed)\n    model = build_model(args, variant, train.R).to(device)\n    teacher = copy.deepcopy(model).eval() if args.ema else None\n    optimizer = torch.optim.AdamW(model.parameters(), lr=args.lr, weight_decay=args.wd)\n    steps_per_epoch = math.ceil(train.G / args.batch_size)\n    total = steps_per_epoch * args.epochs\n    warm = max(1, int(total * 0.05))\n    schedule = torch.optim.lr_scheduler.LambdaLR(\n        optimizer, lambda s: (s + 1) / warm if s < warm else\n        0.5 * (1 + math.cos(math.pi * (s - warm) / max(1, total - warm))) * 0.95 + 0.05,\n    )\n    amp = bool(args.amp) and device.type == "cuda"\n    scaler = torch.amp.GradScaler("cuda", enabled=amp)\n    gm = train.GF.float().mean(0).cpu()\n    gs = train.GF.float().std(0).cpu().clamp_min(1e-6)\n    ema_checkpoint_path = args.output_dir / f"v3_{variant}_ema_seed{seed}.pt"\n    raw_checkpoint_path = args.output_dir / f"v3_{variant}_raw_seed{seed}.pt"\n    record_path = args.output_dir / f"v3_{variant}_dual_seed{seed}.train.json"\n    saved_args = {k: str(v) if isinstance(v, Path) else v for k, v in vars(args).items()}\n    saved_args.update(seeds=[seed], cons=0.0, readout=effective_readout(args, variant))\n    history = []\n    best_ema_score = best_raw_score = float("inf")\n    best_ema_epoch = best_raw_epoch = 0\n    bad = step = 0\n    start = time.time()\n    sampler_generator = torch.Generator(device="cpu").manual_seed(seed)\n    print(f"{variant} seed={seed} | R={train.R} mode={args.struct_mode} | "\n          f"effective batch={args.batch_size}, micro={args.micro_batch_size}", flush=True)\n\n    for epoch in range(1, args.epochs + 1):\n        model.train()\n        permutation = torch.randperm(train.G, generator=sampler_generator)\n        epoch_loss = 0.0\n        for offset in range(0, train.G, args.batch_size):\n            indices = permutation[offset:offset + args.batch_size]\n            optimizer.zero_grad(set_to_none=True)\n            for micro in indices.split(args.micro_batch_size):\n                batch = train.batch(micro, device, gm, gs)\n                weight = batch["B"] / indices.numel()\n                branch_weight = 0.5 if args.aug != "none" else 1.0\n                # Release clean activations BEFORE the augmented forward. The\n                # effective step still averages all examples/branches exactly.\n                with torch.autocast(device.type, dtype=torch.float16, enabled=amp):\n                    clean = model(batch)\n                    clean_loss = F.binary_cross_entropy_with_logits(clean.float(), batch["y"])\n                scaler.scale(weight * branch_weight * clean_loss).backward()\n                epoch_loss += branch_weight * float(clean_loss.detach()) * batch["B"]\n                del clean, clean_loss\n                if args.aug != "none":\n                    with torch.autocast(device.type, dtype=torch.float16, enabled=amp):\n                        noisy = model(batch, aug_mode=args.aug)\n                        noisy_loss = F.binary_cross_entropy_with_logits(noisy.float(), batch["y"])\n                    scaler.scale(weight * 0.5 * noisy_loss).backward()\n                    epoch_loss += 0.5 * float(noisy_loss.detach()) * batch["B"]\n                    del noisy, noisy_loss\n                del batch\n            scaler.unscale_(optimizer)\n            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)\n            previous_scale = scaler.get_scale()\n            scaler.step(optimizer)\n            scaler.update()\n            if scaler.get_scale() >= previous_scale:\n                schedule.step()\n                step += 1\n                if teacher is not None:\n                    decay = min(args.ema_decay, (1 + step) / (10 + step))\n                    with torch.no_grad():\n                        for target, current in zip(teacher.parameters(), model.parameters()):\n                            target.mul_(decay).add_(current.detach(), alpha=1 - decay)\n        # Each weight family chooses its own best validation epoch.  The raw\n        # model and EMA teacher therefore receive identical training updates,\n        # but neither is selected using the other model\'s validation scores.\n        ema_model = teacher if teacher is not None else model\n        raw_model = model\n        truth = val.y.numpy().astype(int)\n\n        def validate(candidate):\n            probabilities = predict(candidate, val, device, gm, gs, bs=args.val_batch_size, amp=amp)\n            swapped = predict(candidate, val, device, gm, gs, bs=args.val_batch_size, amp=amp, aug_mode="swap")\n            metrics = score_probabilities(truth, probabilities, 0.5)\n            clean_loss = validation_loss(probabilities, truth)\n            swap_loss = validation_loss(swapped, truth)\n            score = {"lossaug": clean_loss + swap_loss, "loss": clean_loss,\n                     "acc": -metrics["accuracy"], "auc": -metrics["auc"]}[args.select]\n            thresholds = np.linspace(0.05, 0.95, 91)\n            accuracies = np.array([((probabilities >= threshold) == truth).mean() for threshold in thresholds])\n            threshold = float(np.median(thresholds[accuracies >= accuracies.max() - 1e-9]))\n            return probabilities, clean_loss, swap_loss, score, metrics, threshold\n\n        ema_p, ema_clean, ema_swap, ema_score, ema_metrics, ema_threshold = validate(ema_model)\n        raw_p, raw_clean, raw_swap, raw_score, raw_metrics, raw_threshold = validate(raw_model)\n        if (not math.isfinite(epoch_loss) or not math.isfinite(ema_score)\n                or not math.isfinite(raw_score)):\n            raise FloatingPointError("Non-finite loss; no completed checkpoint will be marked")\n\n        ema_improved = ema_score < best_ema_score - 1e-6\n        raw_improved = raw_score < best_raw_score - 1e-6\n        history.append(dict(\n            epoch=epoch, train_loss=epoch_loss / train.G,\n            ema_val_loss=ema_clean, ema_val_swap_loss=ema_swap,\n            ema_selection_score=ema_score, ema_metrics=ema_metrics,\n            raw_val_loss=raw_clean, raw_val_swap_loss=raw_swap,\n            raw_selection_score=raw_score, raw_metrics=raw_metrics,\n            ema_improved=ema_improved, raw_improved=raw_improved,\n        ))\n\n        # Clone both states before saving.  The counterpart state is retained\n        # only as provenance; `state` is exactly the weight family being tested.\n        current_raw = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}\n        current_ema = {k: v.detach().cpu().clone() for k, v in ema_model.state_dict().items()}\n        common = dict(args=saved_args, R=train.R, gm=gm, gs=gs,\n                      method=f"v3_{variant}", struct_mode=args.struct_mode,\n                      training_complete=False, run_signature=run_signature,\n                      provenance=provenance, dual_selection=True)\n        if ema_improved:\n            best_ema_score, best_ema_epoch = ema_score, epoch\n            atomic_save(dict(\n                state=current_ema, raw_state=current_raw, ema_state=current_ema,\n                thr=ema_threshold, best_epoch=epoch, weight_source="ema",\n                selection_model="ema", selection_score=ema_score, **common,\n            ), ema_checkpoint_path)\n        if raw_improved:\n            best_raw_score, best_raw_epoch = raw_score, epoch\n            atomic_save(dict(\n                state=current_raw, raw_state=current_raw, ema_state=current_ema,\n                thr=raw_threshold, best_epoch=epoch, weight_source="raw",\n                selection_model="raw", selection_score=raw_score, **common,\n            ), raw_checkpoint_path)\n        bad = 0 if (ema_improved or raw_improved) else bad + 1\n        write_json(record_path, dict(\n            complete=False, run_signature=run_signature, provenance=provenance,\n            best_ema_epoch=best_ema_epoch, best_raw_epoch=best_raw_epoch,\n            best_ema_score=best_ema_score, best_raw_score=best_raw_score,\n            history=history,\n        ))\n        print(f"Epoch {epoch:02d} | loss {epoch_loss/train.G:.4f} | "\n              f"EMA val {ema_clean:.4f} ({\'best\' if ema_improved else \'keep\'}) | "\n              f"Raw val {raw_clean:.4f} ({\'best\' if raw_improved else \'keep\'})", flush=True)\n        if bad >= args.patience:\n            break\n\n    for checkpoint_path in (ema_checkpoint_path, raw_checkpoint_path):\n        finished = torch.load(checkpoint_path, map_location="cpu", weights_only=True)\n        finished["training_complete"] = True\n        atomic_save(finished, checkpoint_path)\n    write_json(record_path, dict(\n        complete=True, run_signature=run_signature, provenance=provenance,\n        checkpoint_sha256={"ema": sha256(ema_checkpoint_path), "raw": sha256(raw_checkpoint_path)},\n        best_ema_epoch=best_ema_epoch, best_raw_epoch=best_raw_epoch,\n        best_ema_score=best_ema_score, best_raw_score=best_raw_score,\n        selection=args.select, history=history, train_seconds=time.time() - start,\n    ))\n    print(f"Saved EMA: {ema_checkpoint_path}", flush=True)\n    print(f"Saved Raw: {raw_checkpoint_path}", flush=True)\n    del model, teacher, optimizer, scaler, schedule\n    gc.collect()\n    if device.type == "cuda":\n        torch.cuda.empty_cache()\n\n\ndef main(argv=None):\n    args = build_parser().parse_args(argv)\n    validate_args(args)\n    import torch\n\n    if args.device == "cuda" and not torch.cuda.is_available():\n        raise RuntimeError("Enable a GPU, or explicitly use --device cpu")\n    args.graph_dir, args.output_dir = args.graph_dir.resolve(), args.output_dir.resolve()\n    args.output_dir.mkdir(parents=True, exist_ok=True)\n    domain = next(item for item in TRAINING_DOMAINS.values()\n                  if (item["train_file"], item["val_file"]) == (args.train_file, args.val_file))\n    sources = [REPO / domain["train_source"], REPO / domain["val_source"]]\n    graphs = [args.graph_dir / f"{name}.pkl" for name in (args.train_file, args.val_file)]\n    data_hashes = {str(p): sha256(p) for p in sources + graphs}\n    pending = []\n    for variant in args.variants:\n        for seed in args.seeds:\n            path = args.output_dir / f"v3_{variant}_ema_seed{seed}.pt"\n            record_path = args.output_dir / f"v3_{variant}_dual_seed{seed}.train.json"\n            sig, provenance = signature(args, variant, seed, data_hashes)\n            if args.skip_completed and completed_run(path, record_path, sig):\n                print(f"Skip completed: {path.name}", flush=True)\n            elif (path.exists() or record_path.exists()) and not args.overwrite:\n                raise FileExistsError(f"Incomplete/different run: {path}. Use a new output directory "\n                                      "or --overwrite; a saved best epoch is not proof training finished.")\n            else:\n                pending.append((variant, seed, sig, provenance))\n    if not pending:\n        return\n    train = load_graph(graphs[0], sources[0], args.num_rel, args.struct_mode)\n    val = load_graph(graphs[1], sources[1], train.R, args.struct_mode)\n    for variant, seed, sig, provenance in pending:\n        train_one(args, variant, seed, train, val, sig, provenance)\n\n\nif __name__ == "__main__":\n    main()\n', 'evaluate_v3_quang.py': '"""Evaluate V3 Core checkpoints; never train.\n\nThe orchestrator uses only the standard library. Graph building and inference\nrun in separate processes so the frozen encoder and multiple classifiers never\nremain on the GPU together. Only trusted, locally generated pickle graphs are\nsupported. Checkpoint .zip files are loaded as PyTorch files, not extracted.\n"""\n\nimport argparse\nimport csv\nfrom dataclasses import asdict, dataclass\nimport hashlib\nimport importlib.metadata\nimport json\nfrom pathlib import Path\nimport statistics\nimport subprocess\nimport sys\nfrom zipfile import ZIP_DEFLATED, ZipFile\n\n\nREPO = Path(__file__).resolve().parent\nMODEL_SOURCE = REPO / "model/QuangV3.py"\nDEFAULT_SEEDS = [2021, 2022, 2023, 2024, 2025]\nMETRICS = ("accuracy", "auc", "f1", "recall_ai", "recall_human")\n\n# A training domain fixes both the source train/validation split and which\n# direction Table 3 represents.  Keeping this mapping explicit prevents a\n# GPT-3.5 checkpoint from accidentally being evaluated as HC3 -> GPT-3.5.\nTRAINING_DOMAINS = {\n    "hc3": {\n        "train_file": "hc3_train",\n        "val_file": "hc3_val",\n        "train_source": "data/original_text/benchmarks/hc3/hc3_train.json",\n        "val_source": "data/original_text/benchmarks/hc3/hc3_val.json",\n    },\n    "gpt35": {\n        "train_file": "gpt3.5_mixed_train_split",\n        "val_file": "gpt3.5_mixed_val_split",\n        "train_source": "data/original_text/benchmarks/gpt35_mixed/gpt3.5_mixed_train_split.json",\n        "val_source": "data/original_text/benchmarks/gpt35_mixed/gpt3.5_mixed_val_split.json",\n    },\n}\n\n\n@dataclass(frozen=True)\nclass Case:\n    suite: str\n    condition: str\n    noise_percent: int\n    source: str\n\n    @property\n    def split(self):\n        return Path(self.source).stem\n\n\ndef benchmark_cases(suite="all", train_domain="hc3"):\n    if train_domain not in TRAINING_DOMAINS:\n        raise ValueError(f"Unknown training domain: {train_domain}")\n    if train_domain == "hc3":\n        in_domain = ("data/original_text/benchmarks/hc3/hc3_test.json",\n                     "data/perturbed_text/hc3/lexical_synonym/hc3_test_adj_{rate}_synonym_replace.json")\n        cross_domain = ("data/original_text/benchmarks/gpt35_mixed/gpt3.5_mixed_test_split.json",\n                        "data/perturbed_text/gpt35_mixed/lexical_synonym/"\n                        "gpt3.5_mixed_test_adj_{rate}_synonym_replace.json")\n    else:\n        in_domain = ("data/original_text/benchmarks/gpt35_mixed/gpt3.5_mixed_test_split.json",\n                     "data/perturbed_text/gpt35_mixed/lexical_synonym/"\n                     "gpt3.5_mixed_test_adj_{rate}_synonym_replace.json")\n        cross_domain = ("data/original_text/benchmarks/hc3/hc3_test.json",\n                        "data/perturbed_text/hc3/lexical_synonym/hc3_test_adj_{rate}_synonym_replace.json")\n    table2 = [Case("table2", "Original", 0, in_domain[0])] + [Case(\n        "table2", f"Synonym {rate}%", rate, in_domain[1].format(rate=rate),\n    ) for rate in (5, 10, 20, 30)]\n    table3 = [Case("table3", "Original", 0, cross_domain[0])] + [Case(\n        "table3", f"Synonym {rate}%", rate, cross_domain[1].format(rate=rate),\n    ) for rate in (5, 10, 20, 30)]\n    table4 = [Case(\n        "table4", "Original", 0,\n        "data/original_text/benchmarks/hc3/hc3_test.json",\n    )] + [Case(\n        "table4", f"{kind.capitalize()} 25%", 25,\n        f"data/perturbed_text/hc3/structural_noise/hc3_test_{kind}_25.json",\n    ) for kind in ("insert", "repeat", "replace", "delete")]\n    if suite == "table2":\n        return table2\n    if suite == "table2_table3":\n        return table2 + table3\n    if suite == "full":\n        return table2 + table3 + table4\n    if suite == "table3":\n        return table3\n    if suite == "table4":\n        return table4\n    if suite == "all":\n        return table3 + table4\n    raise ValueError(f"Unknown suite: {suite}")\n\n\ndef sha256(path):\n    digest = hashlib.sha256()\n    with Path(path).open("rb") as stream:\n        for chunk in iter(lambda: stream.read(1024 * 1024), b""):\n            digest.update(chunk)\n    return digest.hexdigest()\n\n\ndef write_json(path, value):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    # A failed write must not leave a report that a resumed run can reuse.\n    temporary = path.with_name(path.name + ".tmp")\n    temporary.write_text(json.dumps(value, indent=2, ensure_ascii=False), encoding="utf-8")\n    temporary.replace(path)\n\n\ndef read_json(path):\n    return json.loads(Path(path).read_text(encoding="utf-8"))\n\n\ndef environment():\n    result = {"python": sys.version.split()[0]}\n    for name in ("torch", "numpy", "spacy", "en-core-web-sm", "transformers", "scikit-learn"):\n        try:\n            result[name] = importlib.metadata.version(name)\n        except importlib.metadata.PackageNotFoundError:\n            result[name] = None\n    return result\n\n\ndef find_checkpoints(root, seeds, variant="quang"):\n    root = Path(root).resolve()\n    if not root.is_dir():\n        raise FileNotFoundError(f"Checkpoint directory does not exist: {root}")\n    if not seeds or len(seeds) != len(set(seeds)):\n        raise ValueError("Provide at least one seed, without duplicates")\n    found = {}\n    for seed in seeds:\n        if variant not in ("quang", "baseline"):\n            raise ValueError(f"Unknown V3 variant: {variant}")\n        stem = (f"p3fin_V1_swap_s{seed}_seed{seed}" if variant == "quang"\n                else f"v3_{variant}_seed{seed}")\n        # Kaggle preserves some uploaded PyTorch checkpoints without a file\n        # extension (for example ``p3fin_V1_swap_s2021_seed2021``).  PyTorch\n        # identifies checkpoints from their contents, not their suffix.\n        paths = sorted(p for p in root.rglob(stem + "*")\n                       if p.is_file() and p.name in (stem, stem + ".pt", stem + ".pth", stem + ".zip"))\n        if not paths:\n            raise FileNotFoundError(f"No {stem} (or .pt/.pth/.zip) under {root}")\n        if len({sha256(path) for path in paths}) != 1:\n            raise ValueError(f"Conflicting checkpoint copies for seed {seed}: {paths}")\n        found[str(seed)] = str(paths[0])\n    return found\n\n\ndef make_config(args):\n    variant = args.variant\n    if variant == "quang" and args.train_domain != "hc3":\n        raise ValueError("Legacy Quang checkpoints only support --train-domain hc3")\n    paths = find_checkpoints(args.checkpoint_dir, args.seeds, variant)\n    files = [MODEL_SOURCE, REPO / "building_graph.py", REPO / "dep_vocab.py",\n             REPO / "dep_vocab.json"]\n    return {\n        "model": "Quang_v3_V1_swap" if variant == "quang" else f"v3_{variant}",\n        "variant": variant, "train_domain": args.train_domain, "seeds": args.seeds,\n        "checkpoint_paths": paths,\n        "checkpoint_sha256": {s: sha256(p) for s, p in paths.items()},\n        "code_sha256": {str(p.relative_to(REPO)): sha256(p) for p in files if p.is_file()},\n        "batch_size": args.batch_size, "device": args.device,\n        "struct_mode": args.struct_mode,\n        "struct_mode_note": ("Legacy checkpoint does not store mode; fixed CLI value, not inferred from test."\n                             if variant == "quang" else "Must match the explicit mode stored during training."),\n        "threshold": 0.5, "ensemble": False, "environment": environment(),\n        "graph_budget_root": args.graph_budget_root,\n    }\n\n\ndef _load_model(config, seed, device):\n    import torch\n    from model.QuangV3 import Net\n\n    path = Path(config["checkpoint_paths"][str(seed)])\n    if sha256(path) != config["checkpoint_sha256"][str(seed)]:\n        raise ValueError(f"Checkpoint changed: {path}")\n    for name, expected_hash in config["code_sha256"].items():\n        if sha256(REPO / name) != expected_hash:\n            raise ValueError(f"Source changed: {name}; start a new run")\n    checkpoint = torch.load(path, map_location="cpu", weights_only=True)\n    if not {"state", "args", "R", "gm", "gs", "thr"} <= checkpoint.keys():\n        raise ValueError(f"Not a Quang V3 checkpoint: {path}")\n    a = checkpoint["args"]\n    expected_train = TRAINING_DOMAINS[config["train_domain"]]["train_file"]\n    if a["train_file"] != expected_train or a["seeds"] != [seed]:\n        raise ValueError(f"Checkpoint training dataset/seed mismatch: {path}")\n    variant = config.get("variant", "quang")\n    if variant == "quang":\n        if a.get("aug") != "swap" or a.get("cons", 0) != 0:\n            raise ValueError(f"Expected V1_swap without consistency: {path}")\n        if checkpoint.get("method") == "v3_baseline":\n            raise ValueError("New ablation checkpoint requires its explicit --variant")\n    else:\n        if checkpoint.get("method") != "v3_baseline":\n            raise ValueError(f"Checkpoint/variant mismatch: {path}")\n        if checkpoint.get("training_complete") is not True:\n            raise ValueError(f"Training did not complete: {path}; saved best epoch is incomplete")\n        if checkpoint.get("struct_mode") != config["struct_mode"]:\n            raise ValueError("--struct-mode differs from the checkpoint training mode")\n    model = Net(\n        int(checkpoint["R"]), d=a["hidden"], nb=a["num_bases"], L=a["layers"],\n        p=a["dropout"], p_in=a["in_dropout"], edge_drop=a["edge_drop"],\n        use_struct=bool(a["struct"]), use_gfeat=bool(a["gfeat"]),\n        readout=a["readout"], aug=a.get("aug", "none"),\n        aug_max=a.get("aug_max", 0.35), mode=config["struct_mode"],\n    )\n    model.load_state_dict(checkpoint["state"], strict=True)\n    return path, checkpoint, model.to(device).eval()\n\n\ndef inspect_checkpoints(config):\n    import gc\n\n    rows = []\n    for seed in config["seeds"]:\n        path, checkpoint, model = _load_model(config, seed, "cpu")\n        row = {\n            "seed": seed, "checkpoint": str(path),\n            "checkpoint_sha256": config["checkpoint_sha256"][str(seed)],\n            "R": int(checkpoint["R"]), "validation_threshold": float(checkpoint["thr"]),\n            "arguments": checkpoint["args"], "struct_mode_used": config["struct_mode"],\n            "parameters": sum(p.numel() for p in model.parameters()),\n            "method": checkpoint.get("method", "Quang_v3_V1_swap"),\n            "best_epoch": checkpoint.get("best_epoch"),\n        }\n        rows.append(row)\n        print(f"Seed {seed}: strict load OK | R={row[\'R\']} | "\n              f"val threshold={row[\'validation_threshold\']:.3f}", flush=True)\n        del checkpoint, model\n        gc.collect()\n    return {"checkpoints": rows, "environment": config["environment"],\n            "struct_mode_note": config["struct_mode_note"]}\n\n\ndef score_probabilities(truth, probabilities, threshold):\n    import numpy as np\n    from sklearn.metrics import accuracy_score, f1_score, roc_auc_score\n\n    pred = (probabilities >= threshold).astype(int)\n    matrix = [[int(((truth == i) & (pred == j)).sum()) for j in (0, 1)] for i in (0, 1)]\n    return {\n        "accuracy": float(accuracy_score(truth, pred)),\n        "auc": float(roc_auc_score(truth, probabilities)),\n        "f1": float(f1_score(truth, pred, zero_division=0)),\n        "recall_ai": matrix[0][0] / sum(matrix[0]),\n        "recall_human": matrix[1][1] / sum(matrix[1]),\n        "predicted_human": int(np.sum(pred)),\n        "confusion_matrix_rows_true_columns_pred": matrix,\n    }\n\n\ndef infer_checkpoint(config, seed, graph_path, expected_samples, signature):\n    import numpy as np\n    import torch\n    from model import QuangV3 as v3\n\n    device = torch.device(config["device"])\n    if device.type == "cuda" and not torch.cuda.is_available():\n        raise RuntimeError("Enable a Kaggle GPU, or explicitly pass --device cpu")\n    v3.set_seed(seed)\n    path, checkpoint, model = _load_model(config, seed, device)\n    relation_count = int(checkpoint["R"])\n    observed = {}\n    original_read = v3.read_raw\n\n    def checked_read(path):\n        emb, edges, relations, labels = original_read(path)\n        if not len(emb) == len(edges) == len(relations) == len(labels) == expected_samples:\n            raise ValueError("Graph/source sample count mismatch; no dropped samples allowed")\n        if not np.isin(labels, [0, 1]).all():\n            raise ValueError("Graph labels must be binary")\n        clipped = total = 0\n        for x, edge, rel in zip(emb, edges, relations):\n            edge, rel = np.asarray(edge), np.asarray(rel)\n            if (x.ndim != 2 or x.shape[0] == 0 or x.shape[1] != 768\n                    or edge.shape != (2, rel.size) or rel.size == 0):\n                raise ValueError("Invalid V1 graph tensor dimensions")\n            if rel.min() < 0 or edge.min() < 0 or edge.max() >= x.shape[0]:\n                raise ValueError("Invalid graph indices")\n            clipped += int((rel >= relation_count).sum())\n            total += int(rel.size)\n        observed.update(clipped_edges=clipped, total_edges=total)\n        if clipped:\n            print(f"WARNING: original V3 code clamps {clipped}/{total} edges to "\n                  f"R-1={relation_count - 1}; verify graph vocabulary", flush=True)\n        return emb, edges, relations, labels\n\n    v3.read_raw = checked_read\n    try:\n        dataset = v3.GraphSet(str(graph_path), R=relation_count,\n                              mode=config["struct_mode"], store_dev="cpu")\n    finally:\n        v3.read_raw = original_read\n    probabilities = v3.predict(\n        model, dataset, device, checkpoint["gm"], checkpoint["gs"],\n        bs=config["batch_size"], amp=bool(checkpoint["args"].get("amp", 0)), aug_mode=None,\n    )\n    truth = dataset.y.cpu().numpy().astype(int)\n    if (len(truth) != expected_samples or len(probabilities) != expected_samples\n            or set(truth.tolist()) != {0, 1} or not np.isfinite(probabilities).all()\n            or not ((probabilities >= 0) & (probabilities <= 1)).all()):\n        raise ValueError("Invalid predictions/labels")\n    threshold = float(checkpoint["thr"])\n    return {\n        "model": config["model"], "train": config["train_domain"], "seed": seed,\n        "split": Path(graph_path).stem, "checkpoint": str(path),\n        "checkpoint_sha256": config["checkpoint_sha256"][str(seed)],\n        "run_signature": signature, "samples": expected_samples, "threshold": 0.5,\n        "batch_size": config["batch_size"], "R": relation_count,\n        "struct_mode_used": config["struct_mode"], **observed,\n        "label_convention": {"0": "ai", "1": "human"},\n        **score_probabilities(truth, probabilities, 0.5),\n        "validation_threshold": threshold,\n        "at_validation_threshold": score_probabilities(truth, probabilities, threshold),\n        "y_true": truth.tolist(), "p_human": probabilities.tolist(),\n    }\n\n\ndef case_signature(config, case, expected_samples):\n    value = {"config": config, "case": asdict(case), "samples": expected_samples,\n             "source_sha256": sha256(REPO / case.source)}\n    return hashlib.sha256(json.dumps(value, sort_keys=True).encode()).hexdigest()\n\n\ndef cached_report_matches(path, signature, seed, checkpoint_hash):\n    try:\n        report = read_json(path)\n        required = set(METRICS) | {"validation_threshold", "at_validation_threshold", "clipped_edges"}\n        return (required <= report.keys()\n                and set(METRICS) <= report["at_validation_threshold"].keys()\n                and report["run_signature"] == signature and report["seed"] == seed\n                and report["checkpoint_sha256"] == checkpoint_hash\n                and report["threshold"] == 0.5\n                and len(report["y_true"]) == len(report["p_human"]) == report["samples"])\n    except (OSError, ValueError, KeyError, TypeError):\n        return False\n\n\ndef report_path(results, case, seed):\n    return results / f"{case.suite}_{seed}_{case.split}.json"\n\n\ndef run_child(*args, cwd=None):\n    subprocess.run([sys.executable, str(Path(__file__).resolve()), *map(str, args)],\n                   cwd=cwd, check=True)\n\n\ndef evaluate_case(case, config, config_path, work_dir, max_graphs, force=False):\n    source = REPO / case.source\n    with source.open(encoding="utf-8") as stream:\n        expected = sum(bool(line.strip()) for line in stream)\n    if expected == 0:\n        raise ValueError(f"Empty source: {source}")\n    signature = case_signature(config, case, expected)\n    results = work_dir / "result"\n    pending = []\n    for seed in config["seeds"]:\n        output = report_path(results, case, seed)\n        if not force and output.exists():\n            if not cached_report_matches(output, signature, seed,\n                                         config["checkpoint_sha256"][str(seed)]):\n                raise ValueError(f"Stale report: {output}. Use a new --work-dir or --force")\n            print(f"Reuse report: {output.name}", flush=True)\n        else:\n            pending.append((seed, output))\n    if not pending:\n        return\n\n    graph_dir = work_dir / "graph_data"\n    graph_dir.mkdir(parents=True, exist_ok=True)\n    cache = graph_dir / f"{case.split}.pkl"\n    metadata = cache.with_suffix(".meta.json")\n    provenance = {"source_sha256": sha256(source), "code_sha256": config["code_sha256"],\n                  "environment": config["environment"]}\n    if cache.exists():\n        if not metadata.exists() or read_json(metadata) != provenance:\n            raise ValueError(f"Unverified/stale graph: {cache}. Use a new --work-dir")\n    else:\n        roots = {graph_dir, REPO / "graph_data", REPO / "TacGia/PRDetect/graph_data",\n                 work_dir.parent / "paper_table4_original/graph_data"}\n        if config.get("graph_budget_root"):\n            roots.add(Path(config["graph_budget_root"]))\n        existing = {p.resolve() for root in roots if root.exists() for p in root.rglob("*.pkl")}\n        if len(existing) >= max_graphs:\n            raise ValueError(f"Graph limit {max_graphs} reached; clean unused caches first: "\n                             + ", ".join(map(str, sorted(existing))))\n        print(f"Build {case.split} ({expected} samples)", flush=True)\n        subprocess.run([sys.executable, str(REPO / "building_graph.py"), "--file", str(source)],\n                       cwd=work_dir, check=True)\n        if not cache.is_file():\n            raise FileNotFoundError(cache)\n        write_json(metadata, provenance)\n    print(f"{case.suite} {case.condition}: cache {cache.stat().st_size / 2**30:.2f} GiB", flush=True)\n    for seed, output in pending:\n        run_child("_test", "--config", config_path, "--seed", seed,\n                  "--graph", cache, "--report", output,\n                  "--expected-samples", expected, "--signature", signature)\n    # Only this runner\'s verified, rebuildable cache; never user source/weights.\n    cache.unlink()\n    metadata.unlink()\n    print(f"Removed rebuildable cache: {cache.name}", flush=True)\n\n\ndef summarize(cases, results, seeds):\n    details, summaries = [], {False: [], True: []}\n    all_reports = {}\n    for case in cases:\n        reports = [read_json(report_path(results, case, seed)) for seed in seeds]\n        labels = reports[0]["y_true"]\n        for seed, report in zip(seeds, reports):\n            if report["seed"] != seed or report["y_true"] != labels:\n                raise ValueError(f"Unpaired labels/seed: {case.split}")\n            for use_val in (False, True):\n                metrics = report["at_validation_threshold"] if use_val else report\n                details.append({\n                    "suite": case.suite, "condition": case.condition,\n                    "noise_percent": case.noise_percent, "seed": seed,\n                    "threshold_mode": "validation" if use_val else "fixed_0.5",\n                    "threshold": report["validation_threshold"] if use_val else 0.5,\n                    **{metric: metrics[metric] for metric in METRICS},\n                    "clipped_edges": report["clipped_edges"],\n                })\n        all_reports[case.suite, case.split] = reports\n        for use_val in (False, True):\n            row = {"suite": case.suite, "condition": case.condition,\n                   "noise_percent": case.noise_percent, "num_seeds": len(seeds)}\n            for metric in METRICS:\n                values = [(r["at_validation_threshold"] if use_val else r)[metric] for r in reports]\n                row[f"{metric}_mean"] = statistics.mean(values)\n                row[f"{metric}_std"] = statistics.stdev(values) if len(values) > 1 else None\n            summaries[use_val].append(row)\n    noise_cases = [c for c in cases if c.suite == "table4" and c.noise_percent == 25]\n    if len(noise_cases) == 4:\n        for use_val in (False, True):\n            row = {"suite": "table4", "condition": "Average (4 noise sets)",\n                   "noise_percent": 25, "num_seeds": len(seeds)}\n            for metric in METRICS:\n                values = []\n                for i in range(len(seeds)):\n                    per_seed = []\n                    for case in noise_cases:\n                        report = all_reports[case.suite, case.split][i]\n                        per_seed.append((report["at_validation_threshold"] if use_val else report)[metric])\n                    values.append(statistics.mean(per_seed))\n                row[f"{metric}_mean"] = statistics.mean(values)\n                row[f"{metric}_std"] = statistics.stdev(values) if len(values) > 1 else None\n            summaries[use_val].append(row)\n    for name, rows in (("per_seed.csv", details),\n                       ("summary_threshold_0.5.csv", summaries[False]),\n                       ("summary_validation_threshold.csv", summaries[True])):\n        with (results / name).open("w", newline="", encoding="utf-8") as stream:\n            writer = csv.DictWriter(stream, fieldnames=list(rows[0]))\n            writer.writeheader()\n            writer.writerows(rows)\n    for row in summaries[False]:\n        spread = f" +/- {100 * row[\'accuracy_std\']:.2f}" if row["accuracy_std"] is not None else ""\n        print(f"{row[\'suite\']:6s} | {row[\'condition\']:25s} | "\n              f"Acc@0.5 {100 * row[\'accuracy_mean\']:.2f}{spread}%", flush=True)\n\n\ndef export_archive(results, archive, cases, seeds, extra_cases=()):\n    """Do not export stale reports from a previous seed/suite in the same folder."""\n    paths = {report_path(results, case, seed) for case in [*cases, *extra_cases] for seed in seeds}\n    paths.update(results / name for name in (\n        "per_seed.csv", "summary_threshold_0.5.csv", "summary_validation_threshold.csv",\n        "checkpoint_metadata.json", "manifest.json",\n    ))\n    with ZipFile(archive, "w", ZIP_DEFLATED) as output:\n        for path in sorted(paths):\n            output.write(path, str(Path("result") / path.relative_to(results)))\n        output.write(Path(__file__), "evaluate_v3_quang.py")\n        output.write(MODEL_SOURCE, "model/QuangV3.py")\n        for relative in ("train_v3.py",):\n            source = REPO / relative\n            if source.is_file():\n                output.write(source, relative)\n\n\ndef run_benchmarks(args):\n    work = Path(args.work_dir).resolve()\n    work.mkdir(parents=True, exist_ok=True)\n    results = work / "result"\n    results.mkdir(exist_ok=True)\n    config = make_config(args)\n    cases = benchmark_cases(args.suite, args.train_domain)\n    for case in cases:\n        if not (REPO / case.source).is_file():\n            raise FileNotFoundError(REPO / case.source)\n    config_path = work / "config.json"\n    write_json(config_path, config)\n    print("No training. Fixed threshold 0.5; validation thresholds reported separately.", flush=True)\n    print(f"Variant {args.variant}, structural mode {args.struct_mode!r}. "\n          + config["struct_mode_note"], flush=True)\n    run_child("inspect", "--config", config_path, "--output", results / "checkpoint_metadata.json")\n\n    execution = list(cases)\n    extra_cases = []\n    if not args.skip_clean_check:\n        clean = next((c for c in cases if c.suite == "table2" and c.noise_percent == 0), None)\n        if clean is None:\n            original = benchmark_cases("table2", args.train_domain)[0]\n            clean = Case("sanity", original.condition, 0, original.source)\n            extra_cases.append(clean)\n        evaluate_case(clean, config, config_path, work, args.max_graphs, args.force)\n        mean = statistics.mean(read_json(report_path(results, clean, s))["accuracy"] for s in args.seeds)\n        print(f"{args.train_domain} in-domain Original mean: {mean:.4f}", flush=True)\n        if (args.variant == "quang" and args.train_domain == "hc3" and sorted(args.seeds) == DEFAULT_SEEDS\n                and abs(mean - 0.9902) > args.clean_tolerance):\n            raise ValueError("Clean HC3 differs substantially from Quang\'s report. Check parser, "\n                             "graph vocabulary and --struct-mode; do not tune on test. "\n                             "--skip-clean-check is an explicit diagnostic override only.")\n        execution = [c for c in cases if c != clean]\n    for case in execution:\n        evaluate_case(case, config, config_path, work, args.max_graphs, args.force)\n    summarize(cases, results, args.seeds)\n    manifest = {**config, "suite": args.suite, "cases": [asdict(c) for c in cases],\n                "sanity_cases": [asdict(c) for c in extra_cases],\n                "clean_check_skipped": args.skip_clean_check,\n                "std_definition": "sample standard deviation across seeds (ddof=1)"}\n    write_json(results / "manifest.json", manifest)\n    name = ("v3_quang_table3_table4_results.zip" if args.variant == "quang" and args.suite == "all"\n            else f"v3_{args.variant}_{args.suite}_results.zip")\n    archive = work / name\n    export_archive(results, archive, cases, args.seeds, extra_cases)\n    print(f"Results: {results}\\nArchive: {archive}", flush=True)\n\n\ndef positive_int(value):\n    value = int(value)\n    if value < 1:\n        raise argparse.ArgumentTypeError("Must be positive")\n    return value\n\n\ndef build_parser():\n    parser = argparse.ArgumentParser(description=__doc__)\n    sub = parser.add_subparsers(dest="command", required=True)\n    for command in ("inspect", "run"):\n        p = sub.add_parser(command)\n        p.add_argument("--variant", choices=("quang", "baseline"), default="quang")\n        p.add_argument("--train-domain", choices=tuple(TRAINING_DOMAINS), default="hc3",\n                       help="Training domain; determines in-domain Table 2 and Table 3 direction")\n        p.add_argument("--checkpoint-dir", default=str(REPO / "model/V3_quang"))\n        p.add_argument("--seeds", type=int, nargs="+", default=DEFAULT_SEEDS)\n        p.add_argument("--batch-size", type=positive_int, default=4)\n        p.add_argument("--struct-mode", choices=("low", "high", "directed"), default="low")\n        p.add_argument("--device", choices=("cuda", "cpu"), default="cuda")\n        p.add_argument("--graph-budget-root", help="Also count caches recursively under this workflow root")\n        if command == "inspect":\n            p.add_argument("--config")\n            p.add_argument("--output")\n        else:\n            p.add_argument("--suite", choices=("all", "table2", "table3", "table4", "table2_table3", "full"), default="all",\n                           help="table2_table3: in-domain Table 2 plus cross-domain Table 3; full also adds Table 4")\n            p.add_argument("--work-dir", default=str(REPO / "result/v3_quang_eval"))\n            p.add_argument("--max-graphs", type=positive_int, default=4)\n            p.add_argument("--force", action="store_true", help="Rerun reports; do not reuse old predictions")\n            p.add_argument("--skip-clean-check", action="store_true")\n            p.add_argument("--clean-tolerance", type=float, default=0.005)\n    p = sub.add_parser("_test", help="Internal isolated inference worker")\n    p.add_argument("--config", required=True)\n    p.add_argument("--seed", type=int, required=True)\n    p.add_argument("--graph", required=True)\n    p.add_argument("--report", required=True)\n    p.add_argument("--expected-samples", type=positive_int, required=True)\n    p.add_argument("--signature", required=True)\n    return parser\n\n\ndef main(argv=None):\n    args = build_parser().parse_args(argv)\n    if args.command == "run":\n        if not 0 <= args.clean_tolerance < 1:\n            raise ValueError("--clean-tolerance must be a probability difference in [0,1)")\n        run_benchmarks(args)\n    elif args.command == "inspect":\n        config = read_json(args.config) if args.config else make_config(args)\n        metadata = inspect_checkpoints(config)\n        if args.output:\n            write_json(args.output, metadata)\n    else:\n        config = read_json(args.config)\n        report = infer_checkpoint(config, args.seed, args.graph, args.expected_samples, args.signature)\n        write_json(args.report, report)\n        print(f"Seed {args.seed} | {report[\'split\']} | Acc@0.5={report[\'accuracy\']:.4f} | "\n              f"Acc@val-thr={report[\'at_validation_threshold\'][\'accuracy\']:.4f}", flush=True)\n\n\nif __name__ == "__main__":\n    main()\n', 'model/QuangV3.py': '"""Inference-only port of Quang\\\'s PRDetect RGCN v3.\n\nSource: docs/PRDetect_RGCN_v3_tong_hop.md (rgcn_plus.py).\nPreserves the checkpoint architecture, legacy relation-half assumptions,\nfeature conversion and clipping. No training or architecture changes.\nUse evaluate_v3_quang.py; mode is not stored in the supplied checkpoints.\n"""\n\nimport math\nimport os\nimport pickle\nimport random\nimport time\n\nimport numpy as np\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\n\nMAXD = 64\nN_GF = 8\n\ndef _load_pickle(path):\n    try:\n        with open(path, "rb") as f:\n            return pickle.load(f)\n    except Exception:\n        return torch.load(path, map_location="cpu", weights_only=False)\n\n\ndef _pick(keys, must, avoid=()):\n    for k in keys:\n        kl = k.lower()\n        if any(m in kl for m in must) and not any(a in kl for a in avoid):\n            return k\n    return None\n\n\ndef read_raw(path):\n    """Trả về (emb_list, ei_list, et_list, y_array) dạng numpy/torch theo từng đồ thị."""\n    d = _load_pickle(path)\n    if isinstance(d, (list, tuple)) and len(d) > 0 and hasattr(d[0], "x"):\n        emb = [g.x for g in d]\n        ei = [g.edge_index for g in d]\n        et = [g.edge_type for g in d]\n        y = [g.y for g in d]\n    elif isinstance(d, dict):\n        keys = list(d.keys())\n        ke = _pick(keys, ["embed"]) or _pick(keys, ["feat", "x"])\n        ki = _pick(keys, ["edge_index"]) or _pick(keys, ["edge"], avoid=["type", "rel", "attr"])\n        kt = _pick(keys, ["edge_type", "etype", "rel"])\n        ky = "y" if "y" in d else _pick(keys, ["label"])\n        miss = [n for n, k in (("embedding", ke), ("edge_index", ki), ("edge_type", kt), ("label", ky)) if k is None]\n        if miss:\n            raise KeyError(f"Không tìm thấy khóa {miss} trong {path}. Các khóa có: {keys}")\n        emb, ei, et, y = d[ke], d[ki], d[kt], d[ky]\n    else:\n        raise TypeError(f"Định dạng pkl không nhận diện được: {type(d)}")\n    y = np.array([float(torch.as_tensor(v).reshape(-1)[0]) for v in y], dtype=np.float32)\n    return list(emb), list(ei), list(et), y\n\n\ndef fwd_mask(et, R, mode):\n    if mode == "directed":\n        return (et >= 0) & (et < R)\n    half = R // 2\n    if mode == "low":\n        return et < half\n    return (et >= half) & (et < 2 * half)\n\n\ndef parent_array(n, src, dst, et, R, mode):\n    m = fwd_mask(et, R, mode) & (src != dst)\n    par = -np.ones(n, dtype=np.int64)\n    if mode == "directed":\n        if np.unique(src[m]).size != int(m.sum()):\n            raise ValueError("Directed mapping requires one non-self parent per node")\n        if ((et < 0) | (et >= R)).any():\n            raise ValueError("Relation ID outside training vocabulary")\n    par[src[m]] = dst[m]\n    if mode == "directed":\n        visited = np.zeros(n, dtype=np.int8)\n        for start in range(n):\n            if visited[start]: continue\n            trail=[]; node=start\n            while node >= 0 and visited[node] == 0:\n                visited[node]=1; trail.append(node); node=par[node]\n            if node >= 0 and visited[node] == 1: raise ValueError("Dependency parent cycle")\n            visited[trail]=2\n    return par, int(m.sum()), np.unique(src[m]).size\n\n\ndef detect_mode(ei_list, et_list, R, n_probe=300):\n    """Chiều \'dependent->head\' là nhóm quan hệ thấp hay cao? Chọn nhóm có mỗi nút <=1 cạnh ra."""\n    best = None\n    for mode in ("low", "high"):\n        ok = tot = 0\n        for ei, et in list(zip(ei_list, et_list))[:n_probe]:\n            ei = np.asarray(ei); et = np.asarray(et)\n            if ei.shape[1] == 0:\n                continue\n            n = int(ei.max()) + 1\n            _, ne, nu = parent_array(n, ei[0], ei[1], et, R, mode)\n            tot += 1\n            ok += int(ne == nu and ne > 0)\n        frac = ok / max(tot, 1)\n        if best is None or frac > best[1]:\n            best = (mode, frac)\n    return best\n\n\ndef struct_features(n, src, dst, et, R, mode):\n    par, _, _ = parent_array(n, src, dst, et, R, mode)\n    ptr = np.arange(n)\n    depth = np.zeros(n, dtype=np.int64)\n    for _ in range(MAXD):\n        nxt = par[ptr]\n        alive = nxt >= 0\n        if not alive.any():\n            break\n        depth += alive\n        ptr = np.where(alive, nxt, ptr)\n    depth = np.minimum(depth, MAXD - 1)\n    haspar = par >= 0\n    nchild = np.bincount(par[haspar], minlength=n) if haspar.any() else np.zeros(n, dtype=np.int64)\n    dist = np.where(haspar, np.abs(np.arange(n) - np.where(haspar, par, 0)), 0)\n    dbucket = np.minimum(np.floor(np.log2(dist + 1)).astype(np.int64), 7)\n    st = np.stack([np.minimum(depth, 31), np.minimum(nchild, 11), dbucket], 1).astype(np.int8)\n    roots = ~haspar\n    n_roots = max(int(roots.sum()), 1)\n    h = np.zeros(n, dtype=np.int64)\n    np.maximum.at(h, ptr, depth)\n    heights = h[roots] if roots.any() else np.array([0])\n    nonleaf = nchild > 0\n    gf = np.array([\n        math.log1p(n), math.log1p(n_roots), n / n_roots, depth.mean(), depth.max(),\n        heights.mean(), np.log1p(dist).mean(), nchild[nonleaf].mean() if nonleaf.any() else 0.0,\n    ], dtype=np.float32)\n    return st, gf\n\n\nclass GraphSet:\n    def __init__(self, path, R=None, mode=None, store_dev="cpu", verbose=True):\n        t0 = time.time()\n        emb, ei, et, y = read_raw(path)\n        G = len(y)\n        if R is None:\n            R = int(max(int(np.asarray(t).max()) for t in et if np.asarray(t).size > 0)) + 1\n        if mode is None:\n            mode, frac = detect_mode(ei, et, R)\n            if verbose:\n                print(f"   [struct] chiều dependent->head = nhóm quan hệ \'{mode}\' (tỉ lệ đồ thị hợp lệ {frac:.3f})")\n            self.mode_frac = frac\n        self.R, self.mode = R, mode\n        nn_, ec = [], []\n        X, SRC, DST, REL, ST, GF = [], [], [], [], [], []\n        for i in range(G):\n            x = emb[i]\n            x = torch.as_tensor(x).to(torch.float16) if not torch.is_tensor(x) else x.to(torch.float16)\n            emb[i] = None  # giải phóng bản float32 sớm\n            n = x.shape[0]\n            e = np.asarray(ei[i]).reshape(2, -1).astype(np.int64)\n            t = np.asarray(et[i]).reshape(-1).astype(np.int64)\n            if t.size and t.max() >= R:\n                t = np.minimum(t, R - 1)\n            st, gf = struct_features(n, e[0], e[1], t, R, mode)\n            X.append(x); nn_.append(n); ec.append(e.shape[1])\n            SRC.append(torch.from_numpy(e[0])); DST.append(torch.from_numpy(e[1])); REL.append(torch.from_numpy(t))\n            ST.append(torch.from_numpy(st)); GF.append(gf)\n        self.nn = torch.tensor(nn_, dtype=torch.long)\n        self.ec = torch.tensor(ec, dtype=torch.long)\n        self.no = torch.cat([torch.zeros(1, dtype=torch.long), self.nn.cumsum(0)])\n        self.eo = torch.cat([torch.zeros(1, dtype=torch.long), self.ec.cumsum(0)])\n        self.X = torch.cat(X); del X\n        self.SRC, self.DST, self.REL = torch.cat(SRC), torch.cat(DST), torch.cat(REL)\n        self.ST = torch.cat(ST).long()\n        self.GF = torch.from_numpy(np.stack(GF))\n        self.y = torch.from_numpy(y)\n        self.G = G\n        self.to(store_dev)\n        if verbose:\n            print(f"   [data] {os.path.basename(path)}: {G} đồ thị, {self.X.shape[0]} nút, R={R}, "\n                  f"y mean={float(self.y.mean()):.3f}, {time.time()-t0:.0f}s")\n\n    def to(self, dev):\n        for k in ("nn", "ec", "no", "eo", "X", "SRC", "DST", "REL", "ST", "GF", "y"):\n            setattr(self, k, getattr(self, k).to(dev))\n        self.dev = dev\n        return self\n\n    def batch(self, idx, dev, gmean=None, gstd=None):\n        idx = idx.to(self.dev)\n        ln, le = self.nn[idx], self.ec[idx]\n        B = idx.numel()\n        cs_n = torch.cumsum(ln, 0) - ln\n        cs_e = torch.cumsum(le, 0) - le\n        node_ids = torch.arange(int(ln.sum()), device=self.dev) - torch.repeat_interleave(cs_n, ln) + torch.repeat_interleave(self.no[idx], ln)\n        edge_ids = torch.arange(int(le.sum()), device=self.dev) - torch.repeat_interleave(cs_e, le) + torch.repeat_interleave(self.eo[idx], le)\n        off = torch.repeat_interleave(cs_n, le)\n        gf = self.GF[idx]\n        if gmean is not None:\n            gf = ((gf - gmean.to(self.dev)) / gstd.to(self.dev)).clamp(-5, 5)\n        return dict(\n            x=self.X[node_ids].to(dev).float(), st=self.ST[node_ids].to(dev),\n            src=(self.SRC[edge_ids] + off).to(dev), dst=(self.DST[edge_ids] + off).to(dev), rel=self.REL[edge_ids].to(dev),\n            bidx=torch.repeat_interleave(torch.arange(B, device=self.dev), ln).to(dev), B=B,\n            gf=gf.to(dev), y=self.y[idx].to(dev),\n        )\n\n\nclass RGCNLayer(nn.Module):\n    """h_i\' = W0 h_i + sum_r sum_{j in N_i^r} (1/c_{i,r}) W_r h_j ,  W_r = sum_b a_rb V_b  (Schlichtkrull 2018, Eq.2-3)."""\n    def __init__(self, d, R, nb):\n        super().__init__()\n        self.d, self.R, self.nb = d, R, nb\n        self.V = nn.Parameter(torch.empty(nb, d, d))\n        self.comp = nn.Parameter(torch.empty(R, nb))\n        self.W0 = nn.Linear(d, d)\n        for b in range(nb):\n            nn.init.xavier_uniform_(self.V.data[b])\n        nn.init.normal_(self.comp, std=1.0 / math.sqrt(nb))\n\n    def forward(self, x, src, dst, rel, norm):\n        N, d, nb = x.size(0), self.d, self.nb\n        H = (x @ self.V.permute(1, 0, 2).reshape(d, nb * d)).view(N, nb, d)\n        coef = self.comp[rel] * norm.unsqueeze(1)            # [E, nb]\n        msg = torch.einsum("ebd,eb->ed", H[src], coef.to(H.dtype)).float()\n        agg = torch.zeros(N, d, device=x.device, dtype=torch.float32).index_add_(0, dst, msg)\n        return self.W0(x).float() + agg\n\n\ndef seg_softmax(s, bidx, B):\n    m = torch.full((B,), -1e30, device=s.device, dtype=s.dtype).scatter_reduce(0, bidx, s, "amax", include_self=True)\n    e = torch.exp(s - m[bidx])\n    den = torch.zeros(B, device=s.device, dtype=s.dtype).index_add_(0, bidx, e)\n    return e / (den[bidx] + 1e-9)\n\n\nclass Net(nn.Module):\n    def __init__(self, R, in_dim=768, d=256, nb=8, L=2, p=0.3, p_in=0.1, edge_drop=0.0,\n                 use_struct=True, use_gfeat=False, readout="pool3", aug="none", aug_max=0.35, mode="low"):\n        super().__init__()\n        self.R, self.d, self.edge_drop = R, d, edge_drop\n        self.aug, self.aug_max, self.mode = aug, aug_max, mode\n        if aug in ("mask", "both"):\n            self.mask_emb = nn.Parameter(torch.randn(in_dim) * 0.02)\n        self.use_struct, self.use_gfeat, self.readout = use_struct, use_gfeat, readout\n        self.drop_in = nn.Dropout(p_in)\n        self.proj = nn.Linear(in_dim, d)\n        self.ln0 = nn.LayerNorm(d)\n        if use_struct:\n            self.e_dep, self.e_chi, self.e_dis = nn.Embedding(32, 16), nn.Embedding(12, 8), nn.Embedding(8, 8)\n            self.sproj = nn.Linear(32, d)\n        self.layers = nn.ModuleList([RGCNLayer(d, R, nb) for _ in range(L)])\n        self.norms = nn.ModuleList([nn.LayerNorm(d) for _ in range(L)])\n        self.drop = nn.Dropout(p)\n        self.gate = nn.Linear(d, 1)\n        ind = 3 * d + (N_GF if use_gfeat else 0)\n        self.mlp = nn.Sequential(nn.Linear(ind, d), nn.GELU(), nn.Dropout(p), nn.Linear(d, 1))\n\n    def _augment(self, x, src, dst, rel, bidx, B, mode, gen):\n        """Nhiễu mức đặc trưng mô phỏng thay từ: mỗi đồ thị chọn tỉ lệ rho~U(0,aug_max) nút;\n        \'swap\' = thay vector bằng vector của một nút khác CÙNG nhãn quan hệ phụ thuộc (giữ vai trò cú pháp),\n        \'mask\' = thay bằng vector mask học được. Cấu trúc cây không đổi."""\n        N, dev = x.size(0), x.device\n        half = self.R // 2\n        node_rel = torch.full((N,), half, device=dev, dtype=torch.long)\n        if self.mode == "low":\n            m = (rel < half) & (src != dst)\n            r = rel\n        else:\n            m = (rel >= half) & (rel < 2 * half) & (src != dst)\n            r = rel - half\n        if self.mode == "directed":\n            node_rel=torch.full((N,), self.R, device=dev, dtype=torch.long)\n            m=(src != dst); r=rel\n        node_rel[src[m]] = r[m]\n        rho = torch.rand(B, device=dev, generator=gen) * self.aug_max\n        sel = torch.rand(N, device=dev, generator=gen) < rho[bidx]\n        coin = torch.rand(N, device=dev, generator=gen) < 0.5\n        if mode == "mask":\n            use_mask = sel\n        elif mode == "both":\n            use_mask = sel & coin\n        else:\n            use_mask = torch.zeros_like(sel)\n        use_swap = sel & ~use_mask\n        key = node_rel.float() * 2.0 + torch.rand(N, device=dev, generator=gen)\n        order = torch.argsort(key)\n        nxt = order.roll(-1)\n        same = node_rel[order] == node_rel[nxt]\n        donor = torch.empty_like(order)\n        donor[order] = torch.where(same, nxt, order)\n        x_new = torch.where(use_swap.unsqueeze(1), x[donor], x)\n        if mode in ("mask", "both"):\n            x_new = torch.where(use_mask.unsqueeze(1), self.mask_emb.to(x.dtype).unsqueeze(0).expand_as(x), x_new)\n        return x_new\n\n    def forward(self, b, aug_mode=None, gen=None):\n        x, src, dst, rel, bidx, B = b["x"], b["src"], b["dst"], b["rel"], b["bidx"], b["B"]\n        N = x.size(0)\n        if aug_mode and aug_mode != "none":\n            x = self._augment(x, src, dst, rel, bidx, B, aug_mode, gen)\n        if self.training and self.edge_drop > 0 and src.numel() > 0:\n            keep = torch.rand(src.numel(), device=src.device) >= self.edge_drop\n            src, dst, rel = src[keep], dst[keep], rel[keep]\n        key = dst * self.R + rel\n        cnt = torch.bincount(key, minlength=N * self.R).float()\n        norm = 1.0 / cnt[key].clamp(min=1.0)\n        h = self.ln0(self.proj(self.drop_in(x)))\n        if self.use_struct:\n            st = b["st"]\n            s = torch.cat([self.e_dep(st[:, 0]), self.e_chi(st[:, 1]), self.e_dis(st[:, 2])], 1)\n            h = h + self.sproj(s)\n        h = F.gelu(h)\n        for lyr, ln in zip(self.layers, self.norms):\n            h = h + self.drop(F.gelu(ln(lyr(h, src, dst, rel, norm))))\n        h = h.float()\n        cn = torch.zeros(B, device=h.device).index_add_(0, bidx, torch.ones(N, device=h.device)).clamp(min=1.0)\n        mean = torch.zeros(B, h.size(1), device=h.device).index_add_(0, bidx, h) / cn.unsqueeze(1)\n        mx = torch.full((B, h.size(1)), -1e30, device=h.device).scatter_reduce(\n            0, bidx.unsqueeze(1).expand_as(h), h, "amax", include_self=True)\n        w = seg_softmax(self.gate(h).squeeze(-1), bidx, B)\n        att = torch.zeros(B, h.size(1), device=h.device).index_add_(0, bidx, h * w.unsqueeze(1))\n        z = [mean, mx, att] + ([b["gf"]] if self.use_gfeat else [])\n        return self.mlp(torch.cat(z, 1)).squeeze(-1)\n\n\ndef set_seed(s):\n    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)\n\n\n@torch.no_grad()\ndef predict(model, ds, dev, gm, gs, bs=64, amp=False, aug_mode=None):\n    model.eval()\n    gen = None\n    if aug_mode:\n        gen = torch.Generator(device=dev)\n        gen.manual_seed(123)\n    out = []\n    for i in range(0, ds.G, bs):\n        idx = torch.arange(i, min(i + bs, ds.G))\n        b = ds.batch(idx, dev, gm, gs)\n        with torch.autocast("cuda", dtype=torch.float16, enabled=amp and dev.type == "cuda"):\n            out.append(torch.sigmoid(model(b, aug_mode=aug_mode, gen=gen).float()).cpu())\n    return torch.cat(out).numpy()\n\n'}
for name,source in SOURCES.items():
    p=REPO/name;p.parent.mkdir(parents=True,exist_ok=True)
    p.write_text(source,encoding='utf-8')
print('Installed original archived sources:',list(SOURCES))


In [ ]:
WORKER='\nimport os, sys, json, traceback, hashlib, random\nfrom pathlib import Path\n\nkind, label, deterministic, steps, device = sys.argv[1:]\nsys.stdout.reconfigure(encoding=\'utf-8\')\nsteps = int(steps)\ndeterministic = deterministic == \'1\'\nif deterministic:\n    os.environ[\'CUBLAS_WORKSPACE_CONFIG\'] = \':4096:8\'\nelse:\n    os.environ.pop(\'CUBLAS_WORKSPACE_CONFIG\', None)\nimport numpy as np\nimport torch\nimport importlib.util\n\nROOT = Path(__file__).resolve().parent\nOUT = ROOT / \'diagnostic_logs\'\nOUT.mkdir(exist_ok=True)\nLOG = OUT / (label + \'.jsonl\')\nLOG.write_text(\'\', encoding=\'utf-8\')\ncurrent_step = 0\nphase = \'startup\'\n\ndef emit(event, **values):\n    record = dict(event=event, step=current_step, phase=phase, **values)\n    with LOG.open(\'a\', encoding=\'utf-8\') as f:\n        f.write(json.dumps(record, allow_nan=False) + \'\\n\')\n\ndef fingerprint(value):\n    h = hashlib.sha256()\n    def visit(v):\n        if isinstance(v, torch.Tensor):\n            a = v.detach().cpu().contiguous()\n            h.update(str((str(a.dtype), tuple(a.shape))).encode())\n            h.update(a.numpy().tobytes())\n        elif isinstance(v, dict):\n            for k in sorted(v):\n                h.update(str(k).encode()); visit(v[k])\n        elif isinstance(v, (list, tuple)):\n            for item in v: visit(item)\n        else:\n            h.update(repr(v).encode())\n    visit(value)\n    return h.hexdigest()\n\ndef rng():\n    return dict(cpu=fingerprint(torch.get_rng_state()),\n                cuda=fingerprint(torch.cuda.get_rng_state()) if device == \'cuda\' else None,\n                numpy=hashlib.sha256(repr(np.random.get_state()).encode()).hexdigest(),\n                python=hashlib.sha256(repr(random.getstate()).encode()).hexdigest())\n\ndef parameters(model):\n    return fingerprint(model.state_dict())\n\ndef trace_init(model):\n    global phase\n    phase = \'initialization\'\n    emit(\'initialization\', parameters=parameters(model), rng=rng())\n\ndef trace_before(model, batch):\n    global current_step, phase\n    current_step += 1\n    phase = \'before_clean\'\n    emit(\'input\', tensors=fingerprint(batch), rng=rng(), parameters=parameters(model))\n\ndef trace_logits(name, logits):\n    global phase\n    phase = name\n    emit(name, logits=fingerprint(logits), values=logits.detach().float().cpu().tolist(), rng=rng())\n\ndef trace_grad(name, model):\n    global phase\n    phase = name\n    gradients = {k: p.grad for k,p in model.named_parameters() if p.grad is not None}\n    emit(name, gradients=fingerprint(gradients), rng=rng())\n\nclass DiagnosticStop(Exception):\n    pass\n\ndef trace_after(model):\n    global phase\n    phase = \'after_update\'\n    emit(\'after_update\', parameters=parameters(model), rng=rng())\n    if current_step >= steps:\n        raise DiagnosticStop()\n\ndef replace_once(source, old, new):\n    if source.count(old) != 1:\n        raise RuntimeError(\'Ambiguous instrumentation: \' + old)\n    return source.replace(old, new, 1)\n\ntry:\n    if device == \'cpu\': torch.set_num_threads(1)\n    if device == \'cuda\' and not torch.cuda.is_available():\n        raise RuntimeError(\'Enable a Kaggle GPU before running this notebook.\')\n    torch.use_deterministic_algorithms(deterministic)\n    torch.backends.cudnn.benchmark = False\n    torch.backends.cudnn.deterministic = deterministic\n    # Both runs use the same precision setting; no AMP, as in the original runs.\n    emit(\'environment\', kind=kind, deterministic=deterministic,\n         python=sys.version, torch=torch.__version__, numpy=np.__version__,\n         cuda=torch.version.cuda, cudnn=torch.backends.cudnn.version(),\n         gpu=torch.cuda.get_device_name(0) if device == \'cuda\' else \'cpu\',\n         cublas=os.environ.get(\'CUBLAS_WORKSPACE_CONFIG\'),\n         tf32_matmul=torch.backends.cuda.matmul.allow_tf32,\n         tf32_cudnn=torch.backends.cudnn.allow_tf32)\n    filename = \'train_v3_dual_best.py\' if kind == \'dual\' else \'train_v3.py\'\n    path = ROOT / filename\n    source = path.read_text(encoding=\'utf-8\')\n    emit(\'source\', trainer_sha256=hashlib.sha256(source.encode()).hexdigest(),\n         model_sha256=hashlib.sha256((ROOT/\'model/QuangV3.py\').read_bytes()).hexdigest())\n    source = replace_once(source, \'    model = build_model(args, variant, train.R).to(device)\',\n                          \'    model = build_model(args, variant, train.R).to(device)\\n    trace_init(model)\')\n    # The cache is in the original first-epoch permutation order. The LR schedule\n    # keeps 6000 training examples and 10 epochs, not the size of the small cache.\n    source = replace_once(source, \'steps_per_epoch = math.ceil(train.G / args.batch_size)\',\n                          \'steps_per_epoch = math.ceil(6000 / args.batch_size)\')\n    source = replace_once(source, \'permutation = torch.randperm(train.G, generator=sampler_generator)\',\n                          \'permutation = torch.arange(train.G)\')\n    source = replace_once(source, \'                    clean = model(batch)\',\n                          \'                    trace_before(model, batch)\\n                    clean = model(batch)\\n                    trace_logits("clean_forward", clean)\')\n    source = replace_once(source, \'                scaler.scale(weight * branch_weight * clean_loss).backward()\',\n                          \'                scaler.scale(weight * branch_weight * clean_loss).backward()\\n                trace_grad("clean_backward", model)\')\n    source = replace_once(source, \'                        noisy = model(batch, aug_mode=args.aug)\',\n                          \'                        noisy = model(batch, aug_mode=args.aug)\\n                        trace_logits("swap_forward", noisy)\')\n    source = replace_once(source, \'                    scaler.scale(weight * 0.5 * noisy_loss).backward()\',\n                          \'                    scaler.scale(weight * 0.5 * noisy_loss).backward()\\n                    trace_grad("swap_backward", model)\')\n    old = \'                            target.mul_(decay).add_(current.detach(), alpha=1 - decay)\'\n    source = replace_once(source, old, old + \'\\n            trace_after(model)\')\n    module = dict(__name__=\'diagnostic_trainer\', __file__=str(path),\n                  trace_init=trace_init, trace_before=trace_before,\n                  trace_logits=trace_logits, trace_grad=trace_grad, trace_after=trace_after)\n    compile(source, str(path), \'exec\')\n    (OUT / (kind + \'_instrumented.py\')).write_text(source, encoding=\'utf-8\')\n    exec(compile(source, str(path), \'exec\'), module)\n    from model.QuangV3 import GraphSet\n    train = GraphSet(str(ROOT/\'diagnostic_graphs/train.pkl\'), R=51, mode=\'directed\', store_dev=\'cpu\')\n    # train_one never reaches validation: it stops immediately after update N.\n    val = train\n    args = module[\'build_parser\']().parse_args([\n        \'--seeds\',\'2026\',\'--batch-size\',\'1\',\'--micro-batch-size\',\'1\',\n        \'--val-batch-size\',\'64\',\'--epochs\',\'10\',\'--patience\',\'10\',\n        \'--struct-mode\',\'directed\',\'--device\',device,\'--ema\',\'1\' if kind == \'dual\' else \'0\',\n        \'--output-dir\',str(OUT)])\n    module[\'validate_args\'](args)\n    module[\'train_one\'](args, \'baseline\', 2026, train, val, \'diagnostic_only\', {})\n    raise RuntimeError(\'The diagnostic unexpectedly reached normal completion.\')\nexcept DiagnosticStop:\n    emit(\'completed\', completed_steps=current_step)\n    print(label, \'completed\', current_step, \'steps\', flush=True)\nexcept Exception as error:\n    emit(\'error\', error_type=type(error).__name__, message=str(error), traceback=traceback.format_exc())\n    print(label, \'ERROR:\', error, flush=True)\n    sys.exit(1)\n'
BUILD="\nimport sys,json,hashlib,pickle\nfrom pathlib import Path\nimport torch\nimport building_graph as builder\nROOT=Path(__file__).resolve().parent\nsteps=int(sys.argv[1])\nassert 1 <= steps <= 6000\nsource=ROOT/'data/original_text/benchmarks/gpt35_mixed/gpt3.5_mixed_train_split.json'\nlines=[line for line in source.read_text(encoding='utf-8').splitlines() if line.strip()]\nassert len(lines)==6000\nids=torch.randperm(6000,generator=torch.Generator().manual_seed(2026))[:steps].tolist()\nselected=[lines[i] for i in ids]\nbuilder.init_models()\nassert not builder.model.training, 'Encoder must be in eval mode, as from_pretrained normally sets it.'\nxs,edges,rels,y=builder.build_graph(selected,return_edge_type=True,bidirectional=False)\nassert len(xs)==steps, 'A build failed; do not compare a shifted sample order.'\ndest=ROOT/'diagnostic_graphs';dest.mkdir(exist_ok=True)\npayload=dict(all_token_embeddings=xs,all_edge_index=edges,all_edge_type=rels,y=y)\nwith (dest/'train.pkl').open('wb') as f:pickle.dump(payload,f)\ncontent={}\nfor key,items in payload.items():\n    h=hashlib.sha256()\n    for value in ([items] if isinstance(items,torch.Tensor) else items):\n        t=value.detach().cpu().contiguous()\n        h.update(str((str(t.dtype),tuple(t.shape))).encode());h.update(t.numpy().tobytes())\n    content[key]=h.hexdigest()\n(dest/'manifest.json').write_text(json.dumps(dict(\n    seed=2026,original_train_size=6000,original_epochs=10,sample_ids=ids,\n    source_sha256=hashlib.sha256(source.read_bytes()).hexdigest(),\n    builder_sha256=hashlib.sha256((ROOT/'building_graph.py').read_bytes()).hexdigest(),\n    encoder_training=builder.model.training,semantic_hashes=content,\n    note='One cache shared by every worker. Only the first N original training samples are built.'\n),indent=2),encoding='utf-8')\nprint('Built shared cache for',steps,'original first-epoch samples')\n"
(REPO/'diagnostic_worker.py').write_text(WORKER,encoding='utf-8')
(REPO/'build_shared.py').write_text(BUILD,encoding='utf-8')
subprocess.run([sys.executable,'build_shared.py',str(STEPS)],cwd=REPO,check=True)
print((REPO/'diagnostic_graphs/manifest.json').read_text(encoding='utf-8'))


In [ ]:
logs=REPO/'diagnostic_logs';logs.mkdir(exist_ok=True)
for deterministic,prefix in [(False,'default'),(True,'det')]:
    for kind in ['raw','dual']:
        for repeat in [1,2]:
            label=f'{prefix}_{kind}_{repeat}'
            print('Running',label,flush=True)
            result=subprocess.run([sys.executable,'diagnostic_worker.py',kind,label,
                                   str(int(deterministic)),str(STEPS),DEVICE],
                                  cwd=REPO,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True)
            (logs/(label+'.console.txt')).write_text(result.stdout,encoding='utf-8')
            print(result.stdout[-2500:])
            if result.returncode:print('Failure captured; continuing remaining diagnostic runs.')


In [ ]:
import json,zipfile
from pathlib import Path
LOGS=REPO/'diagnostic_logs'
def read(name):
    p=LOGS/(name+'.jsonl')
    return [json.loads(line) for line in p.read_text(encoding='utf-8').splitlines()] if p.exists() else []
def compare(left,right):
    a,b=read(left),read(right)
    errors={name:[r for r in rows if r['event']=='error'] for name,rows in [(left,a),(right,b)]}
    events={'initialization','input','clean_forward','clean_backward','swap_forward','swap_backward','after_update'}
    aa=[r for r in a if r['event'] in events];bb=[r for r in b if r['event'] in events]
    completed=all(any(r['event']=='completed' for r in rows) for rows in [a,b])
    first=None
    for x,y in zip(aa,bb):
        if x!=y:
            first=dict(step=x['step'],event_left=x['event'],event_right=y['event'],
                       differing_fields=[k for k in set(x)|set(y) if x.get(k)!=y.get(k)],left=x,right=y)
            break
    if first is None and len(aa)!=len(bb):
        first=dict(reason='Trace lengths differ',left_length=len(aa),right_length=len(bb))
    return dict(left=left,right=right,status=(
        'error_or_incomplete' if not completed else 'identical_for_tested_steps' if first is None else 'diverged'),
        first_difference=first,errors=errors)
pairs=[('default_raw_1','default_raw_2'),('default_dual_1','default_dual_2'),
       ('default_raw_1','default_dual_1'),('det_raw_1','det_raw_2'),
       ('det_dual_1','det_dual_2'),('det_raw_1','det_dual_1')]
summary=dict(steps=STEPS,pairs=[compare(a,b) for a,b in pairs],
    scope='First N optimizer steps before any validation; same cache, original sample order, original LR schedule. Does not prove later epochs identical.',
    interpretation={
      'repeat_run_diverges':'Same trainer differs across fresh runs: check CUDA determinism and first differing stage.',
      'deterministic_error':'Read the saved traceback. An unsupported deterministic operation is not proof of the original cause.',
      'all_equal':'No divergence in the tested prefix. Increase STEPS; this is not a claim that complete runs match.'})
(LOGS/'comparison.json').write_text(json.dumps(summary,indent=2),encoding='utf-8')
for p in summary['pairs']:
    first=p['first_difference'] or {}
    print(p['left'],'vs',p['right'],':',p['status'],'first step:',first.get('step'),'stage:',first.get('event_left'))
archive=WORK/'raw_dual_diagnostic.zip'
with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED) as z:
    for p in LOGS.rglob('*'):
        if p.is_file():z.write(p,'logs/'+p.relative_to(LOGS).as_posix())
    z.write(REPO/'diagnostic_graphs/manifest.json','graph_manifest.json')
    for name in SOURCES:z.write(REPO/name,'sources/'+name)
    for name in ['diagnostic_worker.py','build_shared.py']:
        z.write(REPO/name,'sources/'+name)
print('Download:',archive)
from IPython.display import FileLink,display
display(FileLink(str(archive)))
